# 02. 피처 실험

- 담당: **② 홍시윤 (데이터·피처)** (이 파일은 담당자만 수정)
- 기록 파일: `experiments/log_H_data.csv`

### 할 일
- [ ] 기본 피처(WT=0, 변이=1)와 비교할 새 피처 만들기 (변이 개수, 변이 종류 구분 등)
- [ ] `run_cv()`로 기본 피처보다 오르는지 확인 → 오르면 `src/features.py`로 옮기기
- [ ] 피처 선택은 반드시 Pipeline 안에서 (fold 안에서만 fit)
- [ ] **화요일 12시 피처 확정**

> **규칙:** 전처리는 Pipeline 안에 / test는 보지 않기 / 점수는 `run_cv()`로 / 결과는 `log_result()`로 기록 / 자세한 규칙은 `docs/coding_rules.md`

## 0. 환경 설정 (수정하지 말고 그대로 실행)
Colab은 구글 드라이브 `내 드라이브`에 레포 폴더(`ah07-cancer-subtype`, 안에 `data/` 포함)를 두면 돼요.

In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-dev) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: /Users/admin/Documents/ah07-cancer-subtype | DATA_DIR: /Users/admin/Documents/ah07-cancer-subtype/data


In [2]:
import platform
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import Pipeline

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import basic_transformer, binarize_mutations
from src.features import mutation_count
from src.validation import get_folds, run_cv, balanced_weights
from src.evaluation import macro_f1, per_class_f1, log_result
from src.prediction import make_submission

print("OS:", platform.platform(), "| python", platform.python_version(),
      "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

OS: macOS-15.7.3-arm64-arm-64bit-Mach-O | python 3.14.8 | pandas 3.1.1 | scikit-learn 1.9.1


## 1. 데이터

In [3]:
train, test = load_data(DATA_DIR)   # test는 형식 확인과 최종 추론에만
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y)                 # 팀 공통 fold
print("train", train.shape, "| test", test.shape, "| classes", len(le.classes_))

train (6201, 4386) | test (2546, 4385) | classes 26


## 실험 예시 (복사해서 바꿔 쓰세요)
1. Pipeline에 피처 변환과 모델을 넣고
2. `run_cv()`로 점수를 내고
3. `log_result()`로 기록해요.

In [4]:
# from xgboost import XGBClassifier
# PARAMS = dict(n_estimators=200, learning_rate=0.1, max_depth=6, tree_method="hist", random_state=SEED, n_jobs=-1)
# pipe = Pipeline([("features", basic_transformer()), ("model", XGBClassifier(**PARAMS))])
# cv = run_cv(pipe, X, y, folds=folds)
# per_class_f1(y, cv["oof"], le.classes_).head(10)
# log_result(owner="H_data", notebook="02_features.ipynb", experiment="실험 이름",
#            features="basic(WT=0,변이=1)", model="XGBoost", params=PARAMS, cv=cv, notes="")

=============================공통 코드 끝=============================

## 2. 피처 구성 방법별 실험

### A. 원본 인코딩

In [5]:
# 원본 데이터에서 변이 문자열 추출
mutations = X.stack()

# 결측치와 WT 제거
mutations = mutations.dropna()
mutations = mutations[mutations.ne("WT")]

# 공백으로 구분된 복수 변이 확인
multi_examples = mutations[
    mutations.astype("string").str.contains(r"\s+", regex=True, na=False)
]

print("복수 변이가 기록된 셀 수:", len(multi_examples))
print("\n복수 변이 예시:")
print(multi_examples.head(10).to_string())

복수 변이가 기록된 셀 수: 22026

복수 변이 예시:
2  ITGB5        Q369* I368N
   LAMB3        S622S G827R
   MYH4        E412K R1800C
   RYR1       D3546V G3739G
   TNC          A368V E646K
5  ADAMTS1      C521Y R258R
   ADCY9        G273R V155A
   ALDH3A1       F336L I71T
   ANO1        I758S P700fs
   BCAN         R107H E468E


In [6]:
# A. 유전자-변이 조합 추출 함수
def extract_mutation_pairs(X):
    pairs = []

    # 샘플별로 유전자 컬럼 확인
    for _, row in X.iterrows():
        sample_pairs = []

        for gene, value in row.items():
            # 결측치 또는 WT는 제외
            if pd.isna(value) or value == "WT":
                continue

            # 공백 기준으로 개별 변이 분리
            for mutation in str(value).split():
                sample_pairs.append(f"{gene}_{mutation}")

        pairs.append(sample_pairs)

    return pairs


# 원본 train 데이터에서 유전자-변이 조합 추출
mutation_pairs = extract_mutation_pairs(X)

print("첫 번째 샘플의 변이 조합:")
print(mutation_pairs[0][:10])

print("\n전체 샘플 수:", len(mutation_pairs))

첫 번째 샘플의 변이 조합:
['ABCC2_D623D', 'ADCY2_F736C', 'CD86_P117R', 'EPHB4_R305S', 'FOXC2_N23K', 'GBF1_L192V', 'GLMN_G592G', 'GNAI1_K277K', 'GOLGA4_A699T', 'GPR182_L111L']

전체 샘플 수: 6201


ex) 

ABCC2_D623D → ABCC2 유전자의 D623D 변이

ADCY2_F736C → ADCY2 유전자의 F736C 변이

전체 샘플 수 6,201개 → 원본 train 데이터와 일치

In [7]:
from collections import Counter

import numpy as np
from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin


class MutationMultiHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, min_frequency=5):
        self.min_frequency = min_frequency

    def fit(self, X, y=None):
        # 학습 fold에서만 변이 조합별 등장 빈도 계산
        mutation_pairs = extract_mutation_pairs(X)

        # 같은 샘플에 동일한 변이가 중복되어도 1회만 집계
        counts = Counter(
            pair
            for sample in mutation_pairs
            for pair in set(sample)
        )

        # 학습 fold에서 5개 이상 샘플에 등장한 변이만 유지
        selected = sorted(
            pair for pair, count in counts.items()
            if count >= self.min_frequency
        )

        # 선택된 변이 조합과 피처 인덱스 저장
        self.feature_names_ = selected
        self.feature_to_idx_ = {
            pair: idx for idx, pair in enumerate(selected)
        }

        return self

    def transform(self, X):
        # fit에서 선택된 피처만 사용
        mutation_pairs = extract_mutation_pairs(X)

        rows = []
        cols = []

        for row_idx, sample in enumerate(mutation_pairs):
            for pair in set(sample):
                col_idx = self.feature_to_idx_.get(pair)

                if col_idx is not None:
                    rows.append(row_idx)
                    cols.append(col_idx)

        # 존재하는 변이 조합은 1, 나머지는 0
        data = np.ones(len(rows), dtype=np.float32)

        return csr_matrix(
            (data, (rows, cols)),
            shape=(len(mutation_pairs), len(self.feature_names_)),
            dtype=np.float32
        )

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_names_, dtype=object)

In [8]:
import pandas as pd

# 변환 동작을 확인하기 위한 가상 데이터
X_sample = pd.DataFrame({
    "TP53": [
        "R175H R248Q",
        "R175H",
        "R175H",
        "WT",
        None
    ],
    "EGFR": [
        "L858R",
        "WT",
        "L858R",
        "L858R",
        "WT"
    ]
})

# 예제 데이터이므로 빈도 기준을 2로 설정
test_encoder = MutationMultiHotEncoder(min_frequency=2)

# 예제 데이터로만 학습 및 변환
X_sample_encoded = test_encoder.fit_transform(X_sample)

# 결과 확인
result = pd.DataFrame(
    X_sample_encoded.toarray(),
    columns=test_encoder.get_feature_names_out()
)

display(result)

print("선택된 피처:", test_encoder.get_feature_names_out().tolist())
print("변환 결과 크기:", X_sample_encoded.shape)

,EGFR_L858R,TP53_R175H
0,1.0,1.0
1,0.0,1.0
2,1.0,1.0
3,1.0,0.0
4,0.0,0.0


선택된 피처: ['EGFR_L858R', 'TP53_R175H']
변환 결과 크기: (5, 2)


In [9]:
from xgboost import XGBClassifier
from src.config import SEED

# B·C·D 실험과 동일한 XGBoost 설정
base_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1
)

print("base_model 생성 완료")

base_model 생성 완료


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.base import clone

pipeline_A = Pipeline([
    ("features", MutationMultiHotEncoder(min_frequency=5)),
    ("model", clone(base_model))
])

print("A 모델 설정:", pipeline_A.named_steps["model"].get_params())
print("A 변환기:", pipeline_A.named_steps["features"])

A 모델 설정: {'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': None, 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.1, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 6, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 200, 'n_jobs': -1, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': None, 'tree_method': 'hist', 'validate_parameters': None, 'verbosity': None}
A 변환기: MutationMultiHotEncoder()


In [11]:
# 기존 공통 fold 사용
train_idx, valid_idx = next(iter(folds))

# 첫 번째 학습 fold만 사용해 인코더 학습
encoder_check = MutationMultiHotEncoder(min_frequency=5)
encoder_check.fit(X.iloc[train_idx])

# 학습 fold와 검증 fold 변환
X_train_A = encoder_check.transform(X.iloc[train_idx])
X_valid_A = encoder_check.transform(X.iloc[valid_idx])

print("선택된 피처 수:", len(encoder_check.feature_names_))
print("학습 데이터 크기:", X_train_A.shape)
print("검증 데이터 크기:", X_valid_A.shape)
print("희소행렬 형식:", type(X_train_A).__name__)
print("학습 데이터의 1 개수:", X_train_A.nnz)

선택된 피처 수: 303
학습 데이터 크기: (4961, 303)
검증 데이터 크기: (1240, 303)
희소행렬 형식: csr_matrix
학습 데이터의 1 개수: 3530


In [12]:
from src.data_loader import encode_target

# 암종 레이블을 숫자로 변환
y_encoded, label_encoder = encode_target(y)

print("인코딩된 타깃 크기:", y_encoded.shape)
print("암종 클래스 수:", len(label_encoder.classes_))

인코딩된 타깃 크기: (6201,)
암종 클래스 수: 26


In [13]:
from src.validation import get_folds, run_cv

# 기존 B·C·D와 동일한 교차검증 분할
folds = get_folds(y_encoded)

print("Fold 수:", len(folds))

Fold 수: 5


### B. 이진 인코딩

In [14]:
# B. 이진 인코딩 결과 확인 (WT=0, 변이=1)

X_B = binarize_mutations(X)

print("원본 데이터 크기:", X.shape)
print("이진 인코딩 데이터 크기:", X_B.shape)
print("변환 후 고유값:", np.unique(X_B.to_numpy()))

display(X_B.iloc[:5, :10])

원본 데이터 크기: (6201, 4384)
이진 인코딩 데이터 크기: (6201, 4384)
변환 후 고유값: [0 1]


,A2M,AAAS,AADAT,AARS1,ABAT,ABCA1,ABCA2,ABCA3,ABCA4,ABCA5
0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0


### C. 이진 인코딩 + 변이 개수

In [15]:
# C. 이진 인코딩 + 환자별 변이 유전자 개수 피처 생성

# B 방식의 이진 인코딩 결과 복사
X_C = X_B.copy()

# 환자별 변이 유전자 개수 추가
X_C["n_mutated_genes"] = mutation_count(X)

# 결과 확인
print("B 방식 데이터 크기:", X_B.shape)
print("C 방식 데이터 크기:", X_C.shape)

print("환자별로 변이가 기록된 유전자 수")
display(X_C[["n_mutated_genes"]].head(10))

B 방식 데이터 크기: (6201, 4384)
C 방식 데이터 크기: (6201, 4385)
환자별로 변이가 기록된 유전자 수


,n_mutated_genes
0,25
1,17
2,119
3,7
4,34
5,270
6,4
7,1
8,9
9,20


In [16]:
# C 방식의 변이 유전자 개수 피처 검증

count_check = X_B.sum(axis=1)

print(
    "변이 개수 일치 여부:",
    (count_check == X_C["n_mutated_genes"]).all()
)

print(
    "결측치 개수:",
    X_C["n_mutated_genes"].isna().sum()
)

변이 개수 일치 여부: True
결측치 개수: 0


### D. 이진 인코딩 + 변이 개수 + 변이 유형별 개수

#### D

In [17]:
# D. 변이 유형별 개수 피처 생성을 위한 원본 변이값 확인

# WT와 결측치를 제외한 실제 변이 기록만 추출
mutation_values = X.stack()
mutation_values = mutation_values[mutation_values != "WT"]

print("전체 변이 기록 셀 수:", len(mutation_values))
print("서로 다른 변이 표기 수:", mutation_values.nunique())

print("\n가장 자주 등장하는 변이 표기 TOP 30")
display(mutation_values.value_counts().head(30))

print("\n변이 표기 예시 30개")
print(mutation_values.drop_duplicates().head(30).tolist())

전체 변이 기록 셀 수: 218893
서로 다른 변이 표기 수: 118846

가장 자주 등장하는 변이 표기 TOP 30


R132H     349
V600E     305
E545K     135
H1047R    128
R273C      82
E542K      68
R175H      66
R273H      60
R132C      59
R248Q      49
R213*      45
R248W      45
L42L       39
R282W      34
R196*      32
Y220C      32
K16fs      31
R80*       31
L35L       28
L100L      27
A2A        25
Q61R       25
Q269Q      25
H179R      25
L45L       24
P423P      24
R306*      24
G245S      24
E110K      24
L159L      23
Name: count, dtype: int64


변이 표기 예시 30개
['D623D', 'F736C', 'P117R', 'R305S', 'N23K', 'L192V', 'G592G', 'K277K', 'A699T', 'L111L', 'I136M', 'A219T', 'E237*', 'A746D', 'D465D', 'N1445D', 'T218T', 'G764G', 'E660Q', 'E291A', 'I382M', 'V412A', 'P222R', 'R104H', 'P17R', 'T1115M', 'L1854fs', 'R36H', 'R69Q', 'A372T']


In [18]:
# D. 한 유전자 셀에 여러 변이 표기가 기록된 경우 확인

# WT와 결측치를 제외한 변이 기록 추출
mutation_values = X.stack()
mutation_values = mutation_values[mutation_values != "WT"]

# 공백 기준으로 나뉘는 표기 개수 계산
token_counts = mutation_values.astype(str).str.split().str.len()

print("전체 변이 기록 셀 수:", len(mutation_values))
print("표기가 1개인 셀 수:", (token_counts == 1).sum())
print("표기가 2개 이상인 셀 수:", (token_counts >= 2).sum())
print("한 셀의 최대 표기 개수:", token_counts.max())

print("\n표기 개수별 셀 수")
display(token_counts.value_counts().sort_index().head(64))

print("\n표기가 2개 이상인 셀 예시 20개")
display(mutation_values[token_counts >= 2].head(20))

전체 변이 기록 셀 수: 218893
표기가 1개인 셀 수: 196867
표기가 2개 이상인 셀 수: 22026
한 셀의 최대 표기 개수: 64

표기 개수별 셀 수


1     196867
2      16261
3       3023
4       1399
5        382
6        374
7         79
8        159
9         25
10        82
11        16
12        63
13         6
14        40
15         5
16        28
17         2
18        19
19         1
20        19
21         1
22        12
24         8
26         6
28         3
30         4
32         2
36         2
38         1
39         1
44         1
60         1
64         1
Name: count, dtype: int64


표기가 2개 이상인 셀 예시 20개


2  ITGB5        Q369* I368N
   LAMB3        S622S G827R
   MYH4        E412K R1800C
   RYR1       D3546V G3739G
   TNC          A368V E646K
5  ADAMTS1      C521Y R258R
   ADCY9        G273R V155A
   ALDH3A1       F336L I71T
   ANO1        I758S P700fs
   BCAN         R107H E468E
   DIAPH3        Q924* P44P
   HDAC9         L99L N825N
   IRF2         K64T N167fs
   JAK2       Y435fs N193fs
   MLF1           H22H G49W
   MMP2         G179G A408A
   PAOX         S269S T286T
   PIAS3       V274M L226fs
   REV3L      Y2576F F1486F
   RHOBTB3      S372S T480T
dtype: str

In [19]:
# D. 전체 변이 표기 분리 및 유형별 분류 결과 확인

# 공백 기준으로 개별 변이 표기 분리
tokens = mutation_values.astype(str).str.split().explode()

# 변이 유형 분류
is_nonsense = tokens.str.fullmatch(r"[A-Z]\d+\*")
is_frameshift = tokens.str.contains("fs", case=False, regex=False)

is_substitution = tokens.str.fullmatch(r"[A-Z]\d+[A-Z]")

is_synonymous = (
    is_substitution
    & (tokens.str[0] == tokens.str[-1])
)

is_missense = is_substitution & ~is_synonymous

# 기타/미분류
is_other = ~(
    is_nonsense
    | is_frameshift
    | is_synonymous
    | is_missense
)

# 분류 결과 확인
print("전체 개별 변이 표기 수:", len(tokens))
print("미스센스:", is_missense.sum())
print("동의 변이:", is_synonymous.sum())
print("넌센스:", is_nonsense.sum())
print("프레임시프트 후보:", is_frameshift.sum())
print("기타/미분류:", is_other.sum())

print("\n분류 중복 개수:",
      (is_frameshift & (is_missense | is_synonymous | is_nonsense)).sum())

print("\n기타/미분류 표기 TOP 30")
display(tokens[is_other].value_counts().head(30))

전체 개별 변이 표기 수: 255164
미스센스: 164740
동의 변이: 66883
넌센스: 13289
프레임시프트 후보: 9914
기타/미분류: 338

분류 중복 개수: 0

기타/미분류 표기 TOP 30


*261*             3
*1232*            3
*459*             3
468_469LG>F*      2
312_313QY>HH      2
82_83DA>EI        2
139_140MG>I*      2
292_293DP>ES      2
127_128KL>KL      2
929_930LL>LL      2
435_436EA>DS      2
*399*             2
*271*             2
711_712FL>FL      2
*220*             2
594_595ME>IK      2
*902*             2
*626*             2
*235*             2
1105_1106PL>PL    2
187_188PL>PL      2
1441_1442AL>AL    2
*837*             2
2069_2070WA>*T    2
*455*             2
*194*             2
*141*             2
*924*             2
*335*             2
264_265MG>I*      2
Name: count, dtype: int64

#### 대표적인 기타 미분류의 변이 표기 
| 표기 | 해석 |
| :--- | :--- |
| `*261*` | 종결 코돈이 유지된 것으로 보이는 표기 |
| `312_313Q>HH` | 두 아미노산이 함께 치환된 표기 |
| `127_128KL>KL` | 두 아미노산이 동일하게 유지된 표기 |
| `468_469LG>F*` | 복합 아미노산 변화 및 종결 코돈 포함 |

#### 추가 피처명
| 변수명 | 설명 |
| :--- | :--- |
| `n_missense` | 단일 아미노산 치환 표기 수 |
| `n_synonymous` | 단일 아미노산 동의 변이 표기 수 |
| `n_nonsense` | 단일 아미노산이 종결 코돈으로 바뀐 표기 수 |
| `n_frameshift` | `fs`가 포함된 프레임시프트 후보 표기 수 |
| `n_other` | 나머지 복합 변이 및 미분류 표기 수 |

In [20]:
# D. 환자별 변이 유형 개수 피처 생성 함수

import re
import pandas as pd

def mutation_type_counts(X):
    # 변이 유형별 개수를 저장할 데이터프레임
    feature_names = [
        "n_missense",
        "n_synonymous",
        "n_nonsense",
        "n_frameshift",
        "n_other",
    ]

    result = pd.DataFrame(
        0,
        index=X.index,
        columns=feature_names,
        dtype="int32"
    )

    # WT와 결측치를 제외한 변이 기록 추출
    mutations = X.stack()
    mutations = mutations[mutations != "WT"]

    # 한 셀에 여러 변이가 있으면 공백 기준으로 분리
    tokens = mutations.astype(str).str.split().explode().dropna()

    # 개별 변이 표기를 유형별로 분류
    def classify(token):
        if "fs" in token.lower():
            return "n_frameshift"

        if re.fullmatch(r"[A-Z]\d+\*", token):
            return "n_nonsense"

        if re.fullmatch(r"[A-Z]\d+[A-Z]", token):
            if token[0] == token[-1]:
                return "n_synonymous"
            return "n_missense"

        return "n_other"

    types = tokens.map(classify)

    # 환자별 변이 유형 개수 집계
    counts = pd.crosstab(
        tokens.index.get_level_values(0),
        types
    )

    result = counts.reindex(
        index=X.index,
        columns=feature_names,
        fill_value=0
    ).astype("int32")

    return result

In [21]:
# D. 이진 인코딩 + 변이 개수 + 변이 유형별 개수 생성

type_counts = mutation_type_counts(X)

X_D = pd.concat([X_C, type_counts], axis=1)

print("C 방식 데이터 크기:", X_C.shape)
print("D 방식 데이터 크기:", X_D.shape)

display(type_counts.head(10))

C 방식 데이터 크기: (6201, 4385)
D 방식 데이터 크기: (6201, 4390)


col_0,n_missense,n_synonymous,n_nonsense,n_frameshift,n_other
0,17,7,1,0,0
1,9,5,0,3,0
2,79,39,6,0,0
3,3,3,1,0,0
4,23,11,0,0,0
5,164,81,11,31,0
6,2,1,0,1,0
7,0,1,0,0,0
8,7,2,0,0,0
9,12,6,2,0,0


In [22]:
# D. 변이 유형별 개수 피처의 합계 및 결측치 검증

# 환자별 변이 유형 개수 합계
total_type_counts = type_counts.sum(axis=1)

print("전체 변이 표기 수:", len(tokens))
print("유형별 개수 총합:", total_type_counts.sum())

print("전체 합계 일치 여부:",
      total_type_counts.sum() == len(tokens))

print("결측치 개수:", type_counts.isna().sum().sum())

print("변이 유전자 수보다 변이 표기 수가 적은 환자:",
      (total_type_counts < X_C["n_mutated_genes"]).sum())

print("\n변이 유형별 전체 개수")
display(type_counts.sum())

전체 변이 표기 수: 255164
유형별 개수 총합: 255164
전체 합계 일치 여부: True
결측치 개수: 0
변이 유전자 수보다 변이 표기 수가 적은 환자: 0

변이 유형별 전체 개수


col_0
n_missense      164740
n_synonymous     66883
n_nonsense       13289
n_frameshift      9914
n_other            338
dtype: int64

#### D + Multi-Hot

In [23]:
# D 피처와 Multi-Hot 피처를 결합하는 변환기

from scipy.sparse import hstack, csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin

class DMultiHotTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, min_frequency=5):
        self.min_frequency = min_frequency

    def fit(self, X, y=None):
        # Multi-Hot 피처는 학습 fold에서만 선택
        self.multi_hot_ = MutationMultiHotEncoder(
            min_frequency=self.min_frequency
        )
        self.multi_hot_.fit(X)
        return self

    def transform(self, X):
        # 기존 D 피처 생성
        X_D = make_features_D(X)

        # Multi-Hot 피처 생성
        X_multi = self.multi_hot_.transform(X)

        # 두 피처를 희소행렬로 결합
        return hstack(
            [csr_matrix(X_D), X_multi],
            format="csr"
        )

## 3. 피처 구성별 모델 성능 비교

### 3-1. Pipeline 피처 변환기 구성 및 검증

In [24]:
# B·C·D 피처 구성을 위한 Pipeline 변환 함수

from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

def make_features_B(X):
    return binarize_mutations(X)

def make_features_C(X):
    X_binary = binarize_mutations(X)

    return pd.concat([
        X_binary,
        mutation_count(X)
    ], axis=1)

def make_features_D(X):
    X_binary = binarize_mutations(X)

    return pd.concat([
        X_binary,
        mutation_count(X),
        mutation_type_counts(X)
    ], axis=1)

# Pipeline에서 사용할 변환기
transformer_B = FunctionTransformer(make_features_B)
transformer_C = FunctionTransformer(make_features_C)
transformer_D = FunctionTransformer(make_features_D)

# 기존 결과와 동일하게 생성되는지 확인
print("B 일치:", transformer_B.transform(X).equals(X_B))
print("C 일치:", transformer_C.transform(X).equals(X_C))
print("D 일치:", transformer_D.transform(X).equals(X_D))

B 일치: True
C 일치: True
D 일치: True


### 3-2. 공통 모델 설정

In [25]:
# XGBoost 정상 작동 확인

import xgboost as xgb

print("XGBoost 버전:", xgb.__version__)

XGBoost 버전: 3.4.1


In [26]:
# 팀 공통 Baseline 설정으로 B·C·D Pipeline 생성

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

base_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

pipeline_B = Pipeline([
    ("features", transformer_B),
    ("model", base_model)
])

pipeline_C = Pipeline([
    ("features", transformer_C),
    ("model", base_model)
])

pipeline_D = Pipeline([
    ("features", transformer_D),
    ("model", base_model)
])

print("B:", pipeline_B.named_steps["model"].get_params()["n_estimators"])
print("C:", pipeline_C.named_steps["model"].get_params()["n_estimators"])
print("D:", pipeline_D.named_steps["model"].get_params()["n_estimators"])

B: 200
C: 200
D: 200


### 3-3. A·B·C·D 교차검증 성능 비교

In [27]:
y_encoded, label_encoder = encode_target(y)

In [28]:
# 타깃 인코딩 결과 확인

print("타깃 데이터 크기:", y_encoded.shape)
print("클래스 개수:", len(label_encoder.classes_))
print("인코딩 결과:", y_encoded[:10])

타깃 데이터 크기: (6201,)
클래스 개수: 26
인코딩 결과: [ 8 19 20  9  6 21  2 23 12 21]


#### A 교차검증 성능비교

**min_frequency=5**

In [29]:
# A 방식: 유전자-변이 조합별 Multi-Hot Encoding
# 학습 fold에서 5회 이상 등장한 변이 조합만 사용
# B·C·D와 동일한 모델 및 교차검증 조건

cv_A = run_cv(
    pipeline_A,
    X,
    y_encoded,
    folds=folds
)

print(f"A 방식 CV Macro F1: {cv_A['mean']:.4f} ± {cv_A['std']:.4f}")

fold 1: Macro F1 = 0.1660
fold 2: Macro F1 = 0.1723
fold 3: Macro F1 = 0.1702
fold 4: Macro F1 = 0.1686
fold 5: Macro F1 = 0.1606
CV Macro F1 = 0.1675 (std 0.0041)
A 방식 CV Macro F1: 0.1675 ± 0.0041


**min_frequency=3**

In [30]:
# A 방식: Multi-Hot Encoding (최소 등장 빈도 3회)
# 기존 A(5회)와 동일한 XGBoost 및 5-Fold 사용

pipeline_A3 = Pipeline([
    ("features", MutationMultiHotEncoder(min_frequency=3)),
    ("model", clone(base_model))
])

cv_A3 = run_cv(
    pipeline_A3,
    X,
    y_encoded,
    folds=folds
)

print("\n[A 방식 - 빈도 3회]")
print(f"CV Macro F1: {cv_A3['mean']:.4f} ± {cv_A3['std']:.4f}")
print(f"Fold별 점수: {cv_A3['scores']}")

fold 1: Macro F1 = 0.1685
fold 2: Macro F1 = 0.1752
fold 3: Macro F1 = 0.1698
fold 4: Macro F1 = 0.1719
fold 5: Macro F1 = 0.1664
CV Macro F1 = 0.1704 (std 0.0030)

[A 방식 - 빈도 3회]
CV Macro F1: 0.1704 ± 0.0030
Fold별 점수: [0.168528250853039, 0.1751909432028513, 0.16978987554059533, 0.17192034407679974, 0.16644845136363162]


#### B 교차검증 성능비교

In [31]:
# B. 이진 인코딩 피처의 5-Fold 교차검증

from src.validation import get_folds, run_cv

folds = get_folds(y_encoded)

result_B = run_cv(
    model=pipeline_B,
    X=X,
    y=y_encoded,
    folds=folds
)

print(f"\nB 평균 Macro F1: {result_B['mean']:.4f}")
print(f"B 표준편차: {result_B['std']:.4f}")

fold 1: Macro F1 = 0.3328
fold 2: Macro F1 = 0.3614
fold 3: Macro F1 = 0.3177
fold 4: Macro F1 = 0.3306
fold 5: Macro F1 = 0.3463
CV Macro F1 = 0.3378 (std 0.0149)

B 평균 Macro F1: 0.3378
B 표준편차: 0.0149


#### C 교차검증 성능비교

In [32]:
# C. 이진 인코딩 + 변이 유전자 수 피처의 5-Fold 교차검증

result_C = run_cv(
    model=pipeline_C,
    X=X,
    y=y_encoded,
    folds=folds
)

print(f"\nC 평균 Macro F1: {result_C['mean']:.4f}")
print(f"C 표준편차: {result_C['std']:.4f}")

# B 대비 성능 변화
diff_C = result_C["mean"] - result_B["mean"]
print(f"B 대비 Macro F1 변화: {diff_C:+.4f}")

fold 1: Macro F1 = 0.3750
fold 2: Macro F1 = 0.3872
fold 3: Macro F1 = 0.3582
fold 4: Macro F1 = 0.3826
fold 5: Macro F1 = 0.4108
CV Macro F1 = 0.3827 (std 0.0172)

C 평균 Macro F1: 0.3827
C 표준편차: 0.0172
B 대비 Macro F1 변화: +0.0450


#### D 교차검증 성능비교

**D**

In [33]:
# D. 이진 인코딩 + 변이 유전자 수 + 변이 유형별 개수의 5-Fold 교차검증

result_D = run_cv(
    model=pipeline_D,
    X=X,
    y=y_encoded,
    folds=folds
)

print(f"\nD 평균 Macro F1: {result_D['mean']:.4f}")
print(f"D 표준편차: {result_D['std']:.4f}")

# B 및 C 대비 성능 변화
diff_D_B = result_D["mean"] - result_B["mean"]
diff_D_C = result_D["mean"] - result_C["mean"]

print(f"B 대비 Macro F1 변화: {diff_D_B:+.4f}")
print(f"C 대비 Macro F1 변화: {diff_D_C:+.4f}")

fold 1: Macro F1 = 0.4004
fold 2: Macro F1 = 0.4225
fold 3: Macro F1 = 0.3745
fold 4: Macro F1 = 0.4244
fold 5: Macro F1 = 0.4165
CV Macro F1 = 0.4077 (std 0.0186)

D 평균 Macro F1: 0.4077
D 표준편차: 0.0186
B 대비 Macro F1 변화: +0.0699
C 대비 Macro F1 변화: +0.0249


**D + Multi-Hot (≥5회)**

In [34]:
# D + Multi-Hot (빈도 5회) 피처 결합 검증

# 첫 번째 fold의 학습·검증 데이터 분리
train_idx, valid_idx = folds[0]

X_train = X.iloc[train_idx]
X_valid = X.iloc[valid_idx]

# 학습 데이터에서만 Multi-Hot 피처 선택
transformer_DA5 = DMultiHotTransformer(min_frequency=5)
transformer_DA5.fit(X_train)

# 학습·검증 데이터 변환
X_train_DA5 = transformer_DA5.transform(X_train)
X_valid_DA5 = transformer_DA5.transform(X_valid)

print("기존 D 피처 수:", make_features_D(X_train).shape[1])
print("Multi-Hot 피처 수:", len(transformer_DA5.multi_hot_.feature_names_))
print("결합된 학습 데이터 크기:", X_train_DA5.shape)
print("결합된 검증 데이터 크기:", X_valid_DA5.shape)
print("데이터 형식:", type(X_train_DA5).__name__)

기존 D 피처 수: 4390
Multi-Hot 피처 수: 303
결합된 학습 데이터 크기: (4961, 4693)
결합된 검증 데이터 크기: (1240, 4693)
데이터 형식: csr_matrix


In [35]:
# D + Multi-Hot (빈도 5회) 5-Fold 교차검증
# 기존 B·C·D·A와 동일한 XGBoost 설정 사용

pipeline_DA5 = Pipeline([
    ("features", DMultiHotTransformer(min_frequency=5)),
    ("model", clone(base_model))
])

cv_DA5 = run_cv(
    pipeline_DA5,
    X,
    y_encoded,
    folds=folds
)

print("\n[D + Multi-Hot - 빈도 5회]")
print(f"CV Macro F1: {cv_DA5['mean']:.4f} ± {cv_DA5['std']:.4f}")
print(f"Fold별 점수: {cv_DA5['scores']}")

fold 1: Macro F1 = 0.4152
fold 2: Macro F1 = 0.4515
fold 3: Macro F1 = 0.3756
fold 4: Macro F1 = 0.4304
fold 5: Macro F1 = 0.4240
CV Macro F1 = 0.4193 (std 0.0249)

[D + Multi-Hot - 빈도 5회]
CV Macro F1: 0.4193 ± 0.0249
Fold별 점수: [0.4151510230378493, 0.4514694468807169, 0.37556888708566005, 0.430417463986306, 0.4240293647824316]


**D + Multi-Hot (≥3회)**

In [36]:
# D + Multi-Hot (빈도 3회) 5-Fold 교차검증
# 기존 실험과 동일한 XGBoost 및 Fold 사용

pipeline_DA3 = Pipeline([
    ("features", DMultiHotTransformer(min_frequency=3)),
    ("model", clone(base_model))
])

cv_DA3 = run_cv(
    pipeline_DA3,
    X,
    y_encoded,
    folds=folds
)

print("\n[D + Multi-Hot - 빈도 3회]")
print(f"CV Macro F1: {cv_DA3['mean']:.4f} ± {cv_DA3['std']:.4f}")
print(f"Fold별 점수: {cv_DA3['scores']}")

fold 1: Macro F1 = 0.4159
fold 2: Macro F1 = 0.4470
fold 3: Macro F1 = 0.3713
fold 4: Macro F1 = 0.4334
fold 5: Macro F1 = 0.4317
CV Macro F1 = 0.4199 (std 0.0262)

[D + Multi-Hot - 빈도 3회]
CV Macro F1: 0.4199 ± 0.0262
Fold별 점수: [0.41592680783374003, 0.44697214592238144, 0.37131594842048965, 0.4333505277512567, 0.43171727025757317]


### 3-4. 실험 결과 기록 및 최종 피처 선정

In [37]:
# A·B·C·D 및 A, D + Multi-Hot 전체 실험 성능 비교표

import pandas as pd

comparison = pd.DataFrame({
    "피처 구성": [
        "A: Multi-Hot (≥5회)",
        "A: Multi-Hot (≥3회)",
        "B: 이진 인코딩",
        "C: 이진 + 변이 유전자 수",
        "D: 이진 + 변이 수 + 유형별 개수",
        "D + Multi-Hot (≥5회)",
        "D + Multi-Hot (≥3회)"
    ],

    "피처 수": [
        "Fold별 상이",
        "Fold별 상이",
        X_B.shape[1],
        X_C.shape[1],
        X_D.shape[1],
        "Fold별 상이",
        "Fold별 상이"
    ],

    "Macro F1 평균": [
        cv_A["mean"],
        cv_A3["mean"],
        result_B["mean"],
        result_C["mean"],
        result_D["mean"],
        cv_DA5["mean"],
        cv_DA3["mean"]
    ],

    "Macro F1 표준편차": [
        cv_A["std"],
        cv_A3["std"],
        result_B["std"],
        result_C["std"],
        result_D["std"],
        cv_DA5["std"],
        cv_DA3["std"]
    ]
})

# 기존 B 방식 대비 성능 개선 폭
comparison["B 대비 개선"] = (
    comparison["Macro F1 평균"] - result_B["mean"]
)

# 현재 최고 성능 순으로 정렬
comparison = comparison.sort_values(
    by="Macro F1 평균",
    ascending=False
).reset_index(drop=True)

display(comparison.round(4))

,피처 구성,피처 수,Macro F1 평균,Macro F1 표준편차,B 대비 개선
0,D + Multi-Hot (≥3회),Fold별 상이,0.4199,0.0262,0.0821
1,D + Multi-Hot (≥5회),Fold별 상이,0.4193,0.0249,0.0816
2,D: 이진 + 변이 수 + 유형별 개수,4390,0.4077,0.0186,0.0699
3,C: 이진 + 변이 유전자 수,4385,0.3827,0.0172,0.0450
4,B: 이진 인코딩,4384,0.3378,0.0149,0.0000
5,A: Multi-Hot (≥3회),Fold별 상이,0.1704,0.0030,-0.1674
6,A: Multi-Hot (≥5회),Fold별 상이,0.1675,0.0041,-0.1702


In [38]:
# 전체 피처 실험 결과를 팀 공통 형식으로 CSV에 기록
# 이미 기록된 실험은 중복 저장하지 않음

import pandas as pd

from src.evaluation import log_result
from src.config import EXPERIMENT_DIR

log_path = EXPERIMENT_DIR / "log_H_data.csv"

# 기존 실험 기록 확인
if log_path.exists():
    existing_log = pd.read_csv(log_path)
    recorded_names = set(existing_log["experiment"].dropna())
else:
    recorded_names = set()

model_params = pipeline_B.named_steps["model"].get_params()

experiments = [
    (
        "A_multihot_freq5",
        "유전자-변이 조합 Multi-Hot (빈도 ≥5회)",
        cv_A,
        "학습 Fold에서 5회 이상 등장한 조합만 사용"
    ),
    (
        "A_multihot_freq3",
        "유전자-변이 조합 Multi-Hot (빈도 ≥3회)",
        cv_A3,
        "학습 Fold에서 3회 이상 등장한 조합만 사용"
    ),
    (
        "B_binary",
        "4384개 유전자 이진 인코딩",
        result_B,
        "WT·결측=0, 변이=1"
    ),
    (
        "C_binary_mutation_count",
        "이진 인코딩 + n_mutated_genes",
        result_C,
        "환자별 변이 유전자 수 추가"
    ),
    (
        "D_binary_mutation_types",
        "이진 인코딩 + n_mutated_genes + 변이 유형별 개수 5개",
        result_D,
        "missense, synonymous, nonsense, frameshift, other 개수 추가"
    ),
    (
        "D_multihot_freq5",
        "D 피처 + Multi-Hot (빈도 ≥5회)",
        cv_DA5,
        "D 피처에 학습 Fold 기준 5회 이상 변이 조합 추가"
    ),
    (
        "D_multihot_freq3",
        "D 피처 + Multi-Hot (빈도 ≥3회)",
        cv_DA3,
        "D 피처에 학습 Fold 기준 3회 이상 변이 조합 추가"
    )
]

for experiment, features, cv, notes in experiments:

    if experiment in recorded_names:
        print(f"[건너뜀] {experiment}: 이미 기록됨")
        continue

    params = model_params.copy()

    if "freq5" in experiment:
        params["min_frequency"] = 5
    elif "freq3" in experiment:
        params["min_frequency"] = 3

    log_result(
        owner="H_data",
        notebook="02_features.ipynb",
        experiment=experiment,
        features=features,
        model="XGBoost",
        params=params,
        cv=cv,
        notes=notes
    )

    recorded_names.add(experiment)
    print(f"[기록 완료] {experiment}")

[건너뜀] A_multihot_freq5: 이미 기록됨
[건너뜀] A_multihot_freq3: 이미 기록됨
[건너뜀] B_binary: 이미 기록됨
[건너뜀] C_binary_mutation_count: 이미 기록됨
[건너뜀] D_binary_mutation_types: 이미 기록됨
[건너뜀] D_multihot_freq5: 이미 기록됨
[건너뜀] D_multihot_freq3: 이미 기록됨


In [39]:
# 실험 로그 저장 결과 확인

from src.config import EXPERIMENT_DIR

log_df = pd.read_csv(EXPERIMENT_DIR / "log_H_data.csv")

display(
    log_df[
        ["experiment", "features", "model", "cv_mean", "cv_std"]
    ]
)

,experiment,features,model,cv_mean,cv_std
0,B_binary,4384개 유전자 이진 인코딩,XGBoost,0.3200,0.0111
1,C_binary_mutation_count,이진 인코딩 + n_mutated_genes,XGBoost,0.3663,0.0093
2,D_binary_mutation_types,이진 인코딩 + n_mutated_genes + 변이 유형별 개수 5개,XGBoost,0.3836,0.0028
3,A_multihot_freq5,유전자-변이 조합 Multi-Hot (빈도 ≥5회),XGBoost,0.1590,0.0032
4,A_multihot_freq3,유전자-변이 조합 Multi-Hot (빈도 ≥3회),XGBoost,0.1630,0.0052
5,D_multihot_freq5,D 피처 + Multi-Hot (빈도 ≥5회),XGBoost,0.3867,0.0164
6,D_multihot_freq3,D 피처 + Multi-Hot (빈도 ≥3회),XGBoost,0.3851,0.0153
7,D_amino_acid_pairs_freq16,D + amino acid substitution pairs (min_frequen...,XGBoost,0.4271,0.0192


**최종 피처 선정 및 실험 결과 해석**

총 7개 피처 구성을 동일한 XGBoost 모델과 Stratified 5-Fold 교차검증으로 비교하였다.

- **최고 평균 성능:** D + Multi-Hot (빈도 ≥5회), Macro F1 0.3867
- **가장 안정적인 상위 후보:** D 단독, Macro F1 0.3836 ± 0.0028
- **Multi-Hot 단독:** Macro F1 0.1590~0.1630으로 이진 인코딩 기반 구성보다 낮은 성능을 보였다.
- **D + Multi-Hot 결합:** D 단독 대비 소폭 개선되었으나 Fold별 성능 변동이 증가하였다.

**최종 판단:** D + Multi-Hot (≥5회)를 최고 평균 성능 후보로 선정하고, D 단독을 안정적인 비교 후보로 유지한다. 두 모델의 OOF 예측 확률을 활용해 앙상블 효과를 추가 검증할 수 있다.

In [40]:
# src/features.py 수정 후 정상 작동 확인

import importlib
import src.features

# 수정된 함수 다시 불러오기
importlib.reload(src.features)

# 공통 함수로 변이 유형별 개수 생성
type_counts_shared = src.features.mutation_type_counts(X)

print("피처 크기:", type_counts_shared.shape)
print("결측치 개수:", type_counts_shared.isna().sum().sum())

print("\n변이 유형별 합계:")
print(type_counts_shared.sum())

피처 크기: (6201, 5)
결측치 개수: 0

변이 유형별 합계:
col_0
n_missense      164740
n_synonymous     66883
n_nonsense       13289
n_frameshift      9914
n_other            338
dtype: int64


### 3-5. OOF 및 test 예측 확률 저장

In [41]:
# 예측 저장을 위한 데이터 및 함수 확인
from src.evaluation import save_predictions
from src.data_loader import load_data
import inspect

# 팀 공통 예측 저장 함수의 인자 확인
print("save_predictions 인자:")
print(inspect.signature(save_predictions))

# 팀 공통 데이터 로더 확인
print("\nload_data 인자:")
print(inspect.signature(load_data))

# 현재 사용 중인 데이터 확인
print("\n학습 데이터 크기:", X.shape)
print("타깃 크기:", y_encoded.shape)
print("클래스 수:", len(label_encoder.classes_))

save_predictions 인자:
(owner: str, name: str, oof_proba, test_proba, classes, cv: dict, notes: str = '') -> None

load_data 인자:
(data_dir=PosixPath('/Users/admin/Documents/ah07-cancer-subtype/data'))

학습 데이터 크기: (6201, 4384)
타깃 크기: (6201,)
클래스 수: 26


In [42]:
# 팀 공통 데이터 로더로 test 데이터 불러오기

train, test = load_data()

# 데이터 내용은 확인하지 않고 구조만 확인
print("test 크기:", test.shape)
print("test 컬럼 수:", len(test.columns))

# 학습 데이터와 test의 피처 구성 및 순서 확인
test_features = test.drop(columns=["ID"])

print("학습·test 컬럼 일치:", X.columns.equals(test_features.columns))
print("test 행 수:", len(test_features))

test 크기: (2546, 4385)
test 컬럼 수: 4385
학습·test 컬럼 일치: True
test 행 수: 2546


In [43]:
# test 데이터의 변이 토큰 생성 과정 확인

mutations_test = test_features.stack()
mutations_test = mutations_test[mutations_test != "WT"]

tokens_test = mutations_test.astype(str).str.split().explode()

print("전체 토큰 수:", len(tokens_test))
print("결측 토큰 수:", tokens_test.isna().sum())

print("\n토큰 타입별 개수:")
print(tokens_test.map(type).value_counts())

전체 토큰 수: 337749
결측 토큰 수: 237

토큰 타입별 개수:
<class 'str'>      337512
<class 'float'>       237
Name: count, dtype: int64


In [44]:
# 수정된 전처리 함수가 test 데이터에서 정상 작동하는지 확인

test_type_counts = mutation_type_counts(test_features)

print("피처 크기:", test_type_counts.shape)
print("결측치 개수:", test_type_counts.isna().sum().sum())

피처 크기: (2546, 5)
결측치 개수: 0


In [45]:
# 기존 학습된 D 모델로 test 예측 재시도

test_proba_D = final_D.predict_proba(test_features)

print("OOF 확률 크기:", result_D["oof_proba"].shape)
print("test 확률 크기:", test_proba_D.shape)

NameError: name 'final_D' is not defined

In [ ]:
# 수정 전후 D 학습 피처가 동일한지 확인

X_D_updated = make_features_D(X)

print("피처 크기 일치:", X_D.shape == X_D_updated.shape)
print("컬럼 구성 일치:", X_D.columns.equals(X_D_updated.columns))
print("전체 피처 값 일치:", X_D.equals(X_D_updated))

피처 크기 일치: True
컬럼 구성 일치: True
전체 피처 값 일치: True


In [ ]:
# 전체 7개 실험의 OOF 및 test 예측 확률 저장

from pathlib import Path
from sklearn.base import clone
from src.evaluation import save_predictions
from src.config import EXPERIMENT_DIR

experiments_to_save = [
    ("A_multihot_freq5", pipeline_A, cv_A),
    ("A_multihot_freq3", pipeline_A3, cv_A3),
    ("B_binary", pipeline_B, result_B),
    ("C_binary_mutation_count", pipeline_C, result_C),
    ("D_binary_mutation_types", pipeline_D, result_D),
    ("D_multihot_freq5", pipeline_DA5, cv_DA5),
    ("D_multihot_freq3", pipeline_DA3, cv_DA3),
]

pred_dir = EXPERIMENT_DIR / "preds"

for name, pipeline, cv in experiments_to_save:

    # 저장된 예측 파일이 있는 경우 재학습 생략
    existing_files = list(pred_dir.glob(f"H_data__{name}*"))

    if existing_files:
        print(f"[건너뜀] {name}: 예측 파일 존재")
        continue

    print(f"\n===== {name} =====")

    # 전체 학습 데이터로 최종 모델 학습
    final_model = clone(pipeline)
    final_model.fit(X, y_encoded)

    # test 예측 확률 생성
    test_proba = final_model.predict_proba(test_features)

    # 확률 배열 크기 확인
    assert cv["oof_proba"].shape == (
        len(X), len(label_encoder.classes_)
    )
    assert test_proba.shape == (
        len(test_features), len(label_encoder.classes_)
    )

    # 팀 공통 형식으로 예측 확률 저장
    save_predictions(
        owner="H_data",
        name=name,
        oof_proba=cv["oof_proba"],
        test_proba=test_proba,
        classes=label_encoder.classes_,
        cv=cv,
        notes="Stratified 5-Fold, Macro F1"
    )

    print(f"[저장 완료] {name}")

print("\n전체 실험 예측 저장 확인 완료")

[건너뜀] A_multihot_freq5: 예측 파일 존재
[건너뜀] A_multihot_freq3: 예측 파일 존재
[건너뜀] B_binary: 예측 파일 존재
[건너뜀] C_binary_mutation_count: 예측 파일 존재
[건너뜀] D_binary_mutation_types: 예측 파일 존재
[건너뜀] D_multihot_freq5: 예측 파일 존재
[건너뜀] D_multihot_freq3: 예측 파일 존재

전체 실험 예측 저장 확인 완료


#### 아미노산 치환 쌍 추출 함수 만들기

In [ ]:
import re
from collections import Counter

def amino_acid_substitution_counts(X):
    """환자별 아미노산 치환 쌍의 발생 횟수를 계산"""

    pattern = re.compile(r"^([A-Z])\d+([A-Z])$")

    all_counts = []

    for _, row in X.iterrows():
        counts = Counter()

        for value in row:
            if pd.isna(value) or value == "WT":
                continue

            # 한 셀에 여러 변이가 있는 경우 각각 분리
            for mutation in str(value).split():
                match = pattern.fullmatch(mutation)

                if match:
                    before, after = match.groups()

                    # 동일 아미노산 치환은 제외
                    if before != after:
                        pair = f"{before}>{after}"
                        counts[pair] += 1

        all_counts.append(counts)

    return all_counts

In [ ]:
sample_counts = amino_acid_substitution_counts(X.head(3))

for i, counts in enumerate(sample_counts):
    print(f"환자 {i + 1}: {dict(counts.most_common(10))}")

환자 1: {'P>R': 3, 'A>T': 2, 'I>M': 2, 'F>C': 1, 'R>S': 1, 'N>K': 1, 'L>V': 1, 'A>D': 1, 'N>D': 1, 'E>Q': 1}
환자 2: {'R>H': 2, 'T>M': 1, 'R>Q': 1, 'A>T': 1, 'A>V': 1, 'P>Q': 1, 'M>I': 1, 'K>M': 1}
환자 3: {'E>K': 15, 'P>S': 11, 'D>N': 8, 'S>F': 4, 'A>V': 4, 'R>C': 3, 'G>R': 3, 'G>E': 3, 'R>W': 2, 'R>Q': 2}


#### D + 치환 쌍 Transformer 구현

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix, hstack


class DAminoAcidTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, min_frequency=16):
        self.min_frequency = min_frequency

    def fit(self, X, y=None):
        # 학습 Fold의 환자별 치환 쌍 추출
        sample_counts = amino_acid_substitution_counts(X)

        # 환자별 중복을 제외한 출현 인원 계산
        patient_frequency = Counter(
            pair
            for counts in sample_counts
            for pair in counts.keys()
        )

        # 학습 Fold에서 기준 인원 이상 출현한 치환 쌍만 선택
        self.selected_pairs_ = sorted(
            pair
            for pair, freq in patient_frequency.items()
            if freq >= self.min_frequency
        )

        self.pair_to_idx_ = {
            pair: idx
            for idx, pair in enumerate(self.selected_pairs_)
        }

        return self

    def transform(self, X):
        # 기존 D 피처 생성
        X_D = make_features_D(X)

        # 환자별 치환 쌍 개수
        sample_counts = amino_acid_substitution_counts(X)

        rows, cols, values = [], [], []

        for row_idx, counts in enumerate(sample_counts):
            for pair, count in counts.items():
                col_idx = self.pair_to_idx_.get(pair)

                if col_idx is not None:
                    rows.append(row_idx)
                    cols.append(col_idx)
                    values.append(np.log1p(count))

        X_pairs = csr_matrix(
            (values, (rows, cols)),
            shape=(len(X), len(self.selected_pairs_)),
            dtype=np.float32
        )

        # D 피처 + 치환 쌍 피처 결합
        return hstack(
            [csr_matrix(X_D), X_pairs],
            format="csr"
        )

In [ ]:
train_idx, val_idx = next(iter(folds))

X_train_fold = X.iloc[train_idx]
X_val_fold = X.iloc[val_idx]

aa_transformer = DAminoAcidTransformer(min_frequency=16)

X_train_aa = aa_transformer.fit_transform(X_train_fold)
X_val_aa = aa_transformer.transform(X_val_fold)

print("선택된 치환 쌍 수:", len(aa_transformer.selected_pairs_))
print("학습 데이터:", X_train_aa.shape)
print("검증 데이터:", X_val_aa.shape)
print("결측값 수:", np.isnan(X_train_aa.data).sum())
print("치환 쌍 예시:", aa_transformer.selected_pairs_[:10])

선택된 치환 쌍 수: 158
학습 데이터: (4960, 4548)
검증 데이터: (1241, 4548)
결측값 수: 0
치환 쌍 예시: ['A>D', 'A>E', 'A>G', 'A>P', 'A>S', 'A>T', 'A>V', 'C>F', 'C>G', 'C>R']


#### XGBoost 5-Fold 교차검증

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# D + 아미노산 치환 쌍 피처를 적용한 XGBoost Pipeline
pipeline_DAA = Pipeline([
    ("features", DAminoAcidTransformer(min_frequency=16)),
    ("model", clone(base_model))
])

# 기존 팀 공통 Fold와 평가 함수 사용
cv_DAA = run_cv(
    pipeline_DAA,
    X,
    y_encoded,
    folds=folds
)

# 기존 D 단독 모델과 비교
print("\n=== XGBoost 성능 비교 ===")
print(f"D 단독         : {result_D['mean']:.4f} ± {result_D['std']:.4f}")
print(f"D + 치환 쌍    : {cv_DAA['mean']:.4f} ± {cv_DAA['std']:.4f}")
print(f"성능 차이       : {cv_DAA['mean'] - result_D['mean']:+.4f}")

fold 0: Macro F1 = 0.4178
fold 1: Macro F1 = 0.4600
fold 2: Macro F1 = 0.4025
fold 3: Macro F1 = 0.4333
fold 4: Macro F1 = 0.4218
CV Macro F1 = 0.4271 (std 0.0192)

=== XGBoost 성능 비교 ===
D 단독         : 0.3836 ± 0.0028
D + 치환 쌍    : 0.4271 ± 0.0192
성능 차이       : +0.0435


#### 실험 로그 추가

In [ ]:
from src.evaluation import log_result

log_result(
    owner="H_data",
    notebook="02_features.ipynb",
    experiment="D_amino_acid_pairs_freq16",
    features="D + amino acid substitution pairs (min_frequency=16, log1p)",
    model="XGBoost",
    params=base_model.get_params(),
    cv=cv_DAA,
    notes="Fold 내부 출현 인원 기준 피처 선택, D 대비 Macro F1 +0.0435"
)

print("8번째 실험 로그 저장 완료")

기록: log_H_data.csv ← D_amino_acid_pairs_freq16 (CV 0.4271)
8번째 실험 로그 저장 완료


#### 실험 예측 파일 저장

In [ ]:
from sklearn.base import clone
from src.evaluation import save_predictions
from src.config import EXPERIMENT_DIR

name = "D_amino_acid_pairs_freq16"
pred_dir = EXPERIMENT_DIR / "preds"

# 이미 저장된 파일이 있는지 확인
expected_files = [
    pred_dir / f"H_data__{name}__oof.npy",
    pred_dir / f"H_data__{name}__test.npy",
    pred_dir / f"H_data__{name}__meta.json",
]

if all(path.exists() for path in expected_files):
    print(f"[건너뜀] {name}: 예측 파일 3개 존재")

else:
    # 전체 train으로 최종 모델 학습
    final_model = clone(pipeline_DAA)
    final_model.fit(X, y_encoded)

    # test 예측 확률 생성
    test_proba = final_model.predict_proba(test_features)

    # 데이터 크기 검증
    assert cv_DAA["oof_proba"].shape == (
        len(X), len(label_encoder.classes_)
    )
    assert test_proba.shape == (
        len(test_features), len(label_encoder.classes_)
    )

    # OOF 및 test 예측 저장
    save_predictions(
        owner="H_data",
        name=name,
        oof_proba=cv_DAA["oof_proba"],
        test_proba=test_proba,
        classes=label_encoder.classes_,
        cv=cv_DAA,
        notes="D + amino acid substitution pairs, min_frequency=16, log1p"
    )

    print(f"[저장 완료] {name}")

예측 저장: H_data__D_amino_acid_pairs_freq16 (CV 0.4271) → experiments/preds/
[저장 완료] D_amino_acid_pairs_freq16


### 4-2. 번외 포함 최종 비교표

In [ ]:
# 전체 8개 실험의 최종 성능 비교
final_results = [
    ("A Multi-Hot (5명 이상)", cv_A),
    ("A Multi-Hot (3명 이상)", cv_A3),
    ("B 이진 인코딩", result_B),
    ("C 변이 수 추가", result_C),
    ("D 변이 종류 추가", result_D),
    ("D + Multi-Hot (5명 이상)", cv_DA5),
    ("D + Multi-Hot (3명 이상)", cv_DA3),
    ("D + 아미노산 치환 쌍 (16명 이상)", cv_DAA),
]

final_comparison = pd.DataFrame([
    {
        "피처 구성": name,
        "Macro F1 평균": cv["mean"],
        "Macro F1 표준편차": cv["std"],
    }
    for name, cv in final_results
])

display(
    final_comparison
    .sort_values("Macro F1 평균", ascending=False)
    .reset_index(drop=True)
    .round(4)
)

,피처 구성,Macro F1 평균,Macro F1 표준편차
0,D + 아미노산 치환 쌍 (16명 이상),0.4271,0.0192
1,D + Multi-Hot (5명 이상),0.3867,0.0164
2,D + Multi-Hot (3명 이상),0.3851,0.0153
3,D 변이 종류 추가,0.3836,0.0028
4,C 변이 수 추가,0.3663,0.0093
5,B 이진 인코딩,0.3200,0.0111
6,A Multi-Hot (3명 이상),0.1630,0.0052
7,A Multi-Hot (5명 이상),0.1590,0.0032


**최종 피처 선정**

- 최종 선정: D + 아미노산 치환 쌍 (min_frequency=16)
- CV Macro F1: 0.4271 ± 0.0192
- 기존 D 대비 +0.0435 개선
- 치환 쌍은 학습 Fold 내부에서 선택하여 데이터 누수를 방지

## 4. 번외 실험 추가

### 4-1. D + 아미노산 치환 쌍 피처 실험

#### 아미노산 치환 쌍 추출 함수 만들기

In [ ]:
import re
from collections import Counter

def amino_acid_substitution_counts(X):
    """환자별 아미노산 치환 쌍의 발생 횟수를 계산"""

    pattern = re.compile(r"^([A-Z])\d+([A-Z])$")

    all_counts = []

    for _, row in X.iterrows():
        counts = Counter()

        for value in row:
            if pd.isna(value) or value == "WT":
                continue

            # 한 셀에 여러 변이가 있는 경우 각각 분리
            for mutation in str(value).split():
                match = pattern.fullmatch(mutation)

                if match:
                    before, after = match.groups()

                    # 동일 아미노산 치환은 제외
                    if before != after:
                        pair = f"{before}>{after}"
                        counts[pair] += 1

        all_counts.append(counts)

    return all_counts

In [ ]:
sample_counts = amino_acid_substitution_counts(X.head(3))

for i, counts in enumerate(sample_counts):
    print(f"환자 {i + 1}: {dict(counts.most_common(10))}")

환자 1: {'P>R': 3, 'A>T': 2, 'I>M': 2, 'F>C': 1, 'R>S': 1, 'N>K': 1, 'L>V': 1, 'A>D': 1, 'N>D': 1, 'E>Q': 1}
환자 2: {'R>H': 2, 'T>M': 1, 'R>Q': 1, 'A>T': 1, 'A>V': 1, 'P>Q': 1, 'M>I': 1, 'K>M': 1}
환자 3: {'E>K': 15, 'P>S': 11, 'D>N': 8, 'S>F': 4, 'A>V': 4, 'R>C': 3, 'G>R': 3, 'G>E': 3, 'R>W': 2, 'R>Q': 2}


#### D + 치환 쌍 Transformer 구현

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix, hstack


class DAminoAcidTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, min_frequency=16):
        self.min_frequency = min_frequency

    def fit(self, X, y=None):
        # 학습 Fold의 환자별 치환 쌍 추출
        sample_counts = amino_acid_substitution_counts(X)

        # 환자별 중복을 제외한 출현 인원 계산
        patient_frequency = Counter(
            pair
            for counts in sample_counts
            for pair in counts.keys()
        )

        # 학습 Fold에서 기준 인원 이상 출현한 치환 쌍만 선택
        self.selected_pairs_ = sorted(
            pair
            for pair, freq in patient_frequency.items()
            if freq >= self.min_frequency
        )

        self.pair_to_idx_ = {
            pair: idx
            for idx, pair in enumerate(self.selected_pairs_)
        }

        return self

    def transform(self, X):
        # 기존 D 피처 생성
        X_D = make_features_D(X)

        # 환자별 치환 쌍 개수
        sample_counts = amino_acid_substitution_counts(X)

        rows, cols, values = [], [], []

        for row_idx, counts in enumerate(sample_counts):
            for pair, count in counts.items():
                col_idx = self.pair_to_idx_.get(pair)

                if col_idx is not None:
                    rows.append(row_idx)
                    cols.append(col_idx)
                    values.append(np.log1p(count))

        X_pairs = csr_matrix(
            (values, (rows, cols)),
            shape=(len(X), len(self.selected_pairs_)),
            dtype=np.float32
        )

        # D 피처 + 치환 쌍 피처 결합
        return hstack(
            [csr_matrix(X_D), X_pairs],
            format="csr"
        )

In [ ]:
train_idx, val_idx = next(iter(folds))

X_train_fold = X.iloc[train_idx]
X_val_fold = X.iloc[val_idx]

aa_transformer = DAminoAcidTransformer(min_frequency=16)

X_train_aa = aa_transformer.fit_transform(X_train_fold)
X_val_aa = aa_transformer.transform(X_val_fold)

print("선택된 치환 쌍 수:", len(aa_transformer.selected_pairs_))
print("학습 데이터:", X_train_aa.shape)
print("검증 데이터:", X_val_aa.shape)
print("결측값 수:", np.isnan(X_train_aa.data).sum())
print("치환 쌍 예시:", aa_transformer.selected_pairs_[:10])

선택된 치환 쌍 수: 158
학습 데이터: (4960, 4548)
검증 데이터: (1241, 4548)
결측값 수: 0
치환 쌍 예시: ['A>D', 'A>E', 'A>G', 'A>P', 'A>S', 'A>T', 'A>V', 'C>F', 'C>G', 'C>R']


#### XGBoost 5-Fold 교차검증

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# D + 아미노산 치환 쌍 피처를 적용한 XGBoost Pipeline
pipeline_DAA = Pipeline([
    ("features", DAminoAcidTransformer(min_frequency=16)),
    ("model", clone(base_model))
])

# 기존 팀 공통 Fold와 평가 함수 사용
cv_DAA = run_cv(
    pipeline_DAA,
    X,
    y_encoded,
    folds=folds
)

# 기존 D 단독 모델과 비교
print("\n=== XGBoost 성능 비교 ===")
print(f"D 단독         : {result_D['mean']:.4f} ± {result_D['std']:.4f}")
print(f"D + 치환 쌍    : {cv_DAA['mean']:.4f} ± {cv_DAA['std']:.4f}")
print(f"성능 차이       : {cv_DAA['mean'] - result_D['mean']:+.4f}")

fold 0: Macro F1 = 0.4178
fold 1: Macro F1 = 0.4600
fold 2: Macro F1 = 0.4025
fold 3: Macro F1 = 0.4333
fold 4: Macro F1 = 0.4218
CV Macro F1 = 0.4271 (std 0.0192)

=== XGBoost 성능 비교 ===
D 단독         : 0.3836 ± 0.0028
D + 치환 쌍    : 0.4271 ± 0.0192
성능 차이       : +0.0435


#### 실험 로그 추가

In [ ]:
from src.evaluation import log_result

log_result(
    owner="H_data",
    notebook="02_features.ipynb",
    experiment="D_amino_acid_pairs_freq16",
    features="D + amino acid substitution pairs (min_frequency=16, log1p)",
    model="XGBoost",
    params=base_model.get_params(),
    cv=cv_DAA,
    notes="Fold 내부 출현 인원 기준 피처 선택, D 대비 Macro F1 +0.0435"
)

print("8번째 실험 로그 저장 완료")

기록: log_H_data.csv ← D_amino_acid_pairs_freq16 (CV 0.4271)
8번째 실험 로그 저장 완료


#### 실험 예측 파일 저장

In [ ]:
from sklearn.base import clone
from src.evaluation import save_predictions
from src.config import EXPERIMENT_DIR

name = "D_amino_acid_pairs_freq16"
pred_dir = EXPERIMENT_DIR / "preds"

# 이미 저장된 파일이 있는지 확인
expected_files = [
    pred_dir / f"H_data__{name}__oof.npy",
    pred_dir / f"H_data__{name}__test.npy",
    pred_dir / f"H_data__{name}__meta.json",
]

if all(path.exists() for path in expected_files):
    print(f"[건너뜀] {name}: 예측 파일 3개 존재")

else:
    # 전체 train으로 최종 모델 학습
    final_model = clone(pipeline_DAA)
    final_model.fit(X, y_encoded)

    # test 예측 확률 생성
    test_proba = final_model.predict_proba(test_features)

    # 데이터 크기 검증
    assert cv_DAA["oof_proba"].shape == (
        len(X), len(label_encoder.classes_)
    )
    assert test_proba.shape == (
        len(test_features), len(label_encoder.classes_)
    )

    # OOF 및 test 예측 저장
    save_predictions(
        owner="H_data",
        name=name,
        oof_proba=cv_DAA["oof_proba"],
        test_proba=test_proba,
        classes=label_encoder.classes_,
        cv=cv_DAA,
        notes="D + amino acid substitution pairs, min_frequency=16, log1p"
    )

    print(f"[저장 완료] {name}")

예측 저장: H_data__D_amino_acid_pairs_freq16 (CV 0.4271) → experiments/preds/
[저장 완료] D_amino_acid_pairs_freq16


#### 번외 포함 비교표

In [ ]:
# 전체 8개 실험의 최종 성능 비교
final_results = [
    ("A Multi-Hot (5명 이상)", cv_A),
    ("A Multi-Hot (3명 이상)", cv_A3),
    ("B 이진 인코딩", result_B),
    ("C 변이 수 추가", result_C),
    ("D 변이 종류 추가", result_D),
    ("D + Multi-Hot (5명 이상)", cv_DA5),
    ("D + Multi-Hot (3명 이상)", cv_DA3),
    ("D + 아미노산 치환 쌍 (16명 이상)", cv_DAA),
]

final_comparison = pd.DataFrame([
    {
        "피처 구성": name,
        "Macro F1 평균": cv["mean"],
        "Macro F1 표준편차": cv["std"],
    }
    for name, cv in final_results
])

display(
    final_comparison
    .sort_values("Macro F1 평균", ascending=False)
    .reset_index(drop=True)
    .round(4)
)

,피처 구성,Macro F1 평균,Macro F1 표준편차
0,D + 아미노산 치환 쌍 (16명 이상),0.4271,0.0192
1,D + Multi-Hot (5명 이상),0.3867,0.0164
2,D + Multi-Hot (3명 이상),0.3851,0.0153
3,D 변이 종류 추가,0.3836,0.0028
4,C 변이 수 추가,0.3663,0.0093
5,B 이진 인코딩,0.3200,0.0111
6,A Multi-Hot (3명 이상),0.1630,0.0052
7,A Multi-Hot (5명 이상),0.1590,0.0032


**피처 선정**

- 최종 선정: D + 아미노산 치환 쌍 (min_frequency=16)
- CV Macro F1: 0.4271 ± 0.0192
- 기존 D 대비 +0.0435 개선
- 치환 쌍은 학습 Fold 내부에서 선택하여 데이터 누수를 방지

### 4-2. E: 아미노산 치환 쌍 + 유전자별 끊김형 피처 검증 (LogisticRegression)

#### E 실험

In [ ]:
import numpy as np
import pandas as pd

from src.features import (
    mutation_count,
    mutation_type_counts,
    amino_acid_pair_counts,
    gene_truncation_flags,
)
from src.preprocessing import binarize_mutations


def make_features_E(X):
    X_binary = binarize_mutations(X)

    X_count = np.log1p(mutation_count(X))
    X_types = np.log1p(mutation_type_counts(X))
    X_pairs = np.log1p(amino_acid_pair_counts(X))
    X_trunc = gene_truncation_flags(X)

    return pd.concat(
        [X_binary, X_count, X_types, X_pairs, X_trunc],
        axis=1
    )


X_E_sample = make_features_E(X.head(3))

print("피처 크기:", X_E_sample.shape)
print("결측값 수:", X_E_sample.isna().sum().sum())
print("중복 컬럼 수:", X_E_sample.columns.duplicated().sum())

피처 크기: (3, 9174)
결측값 수: 0
중복 컬럼 수: 0


In [ ]:
import importlib
import src.features

importlib.reload(src.features)

print("새 함수 존재:", hasattr(src.features, "amino_acid_pair_counts"))
print("끊김형 함수 존재:", hasattr(src.features, "gene_truncation_flags"))

새 함수 존재: True
끊김형 함수 존재: True


In [ ]:
import importlib
import src.validation

importlib.reload(src.validation)

from src.validation import get_folds, run_cv

folds = get_folds(y_encoded)

print("Fold 수:", len(folds))
print("첫 번째 Fold 학습 샘플 수:", len(folds[0][0]))
print("첫 번째 Fold 검증 샘플 수:", len(folds[0][1]))
print("CV 함수 위치:", src.validation.__file__)

Fold 수: 5
첫 번째 Fold 학습 샘플 수: 4961
첫 번째 Fold 검증 샘플 수: 1240
CV 함수 위치: /Users/admin/Documents/ah07-cancer-subtype/src/validation.py


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import csr_matrix
from src.validation import get_folds, run_cv

# 피처 생성 후 희소행렬로 변환
def make_features_E_sparse(X):
    return csr_matrix(make_features_E(X).to_numpy(dtype=np.float32))

pipeline_E_LR = Pipeline([
    ("features", FunctionTransformer(make_features_E_sparse)),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])

# 최신 팀 공통 Fold
folds = get_folds(y_encoded)

cv_E_LR = run_cv(
    pipeline_E_LR,
    X,
    y_encoded,
    folds=folds
)

print("=== 새 피처 E + LogisticRegression ===")
print(f"CV Macro F1: {cv_E_LR['mean']:.4f}")
print(f"표준편차: {cv_E_LR['std']:.4f}")
print("Fold별 점수:", cv_E_LR["scores"])

fold 1: Macro F1 = 0.4934
fold 2: Macro F1 = 0.4797
fold 3: Macro F1 = 0.4758
fold 4: Macro F1 = 0.5161
fold 5: Macro F1 = 0.4947
CV Macro F1 = 0.4919 (std 0.0141)
=== 새 피처 E + LogisticRegression ===
CV Macro F1: 0.4919
표준편차: 0.0141
Fold별 점수: [0.49335575210045995, 0.4797296792353784, 0.4758460427842044, 0.5160511136338943, 0.49469242822842524]


#### 실험 로그 추가

In [ ]:
import pandas as pd
from src.evaluation import log_result
from src.config import EXPERIMENT_DIR

experiment_name = "E_aa_pairs_truncation_group_lr"

log_path = EXPERIMENT_DIR / "log_H_data.csv"

if log_path.exists():
    existing = pd.read_csv(log_path)
    already_logged = experiment_name in existing["experiment"].values
else:
    already_logged = False

if already_logged:
    print("이미 기록된 실험입니다.")
else:
    log_result(
        owner="H_data",
        notebook="02_features.ipynb",
        experiment=experiment_name,
        features="Binary + log1p(mutation count, types, AA pairs) + gene truncation",
        model="LogisticRegression",
        params=pipeline_E_LR.named_steps["model"].get_params(),
        cv=cv_E_LR,
        notes="New grouped 5-fold, all AA pairs, all gene truncation flags, no frequency selection"
    )
    print("9번째 실험 로그 저장 완료")

기록: log_H_data.csv ← E_aa_pairs_truncation_group_lr (CV 0.4919)
9번째 실험 로그 저장 완료


#### 실험 예측 파일 저장

In [ ]:
from pathlib import Path
from sklearn.base import clone
from src.evaluation import save_predictions
from src.config import EXPERIMENT_DIR

name = "E_aa_pairs_truncation_group_lr"
pred_dir = EXPERIMENT_DIR / "preds"

expected_files = [
    pred_dir / f"H_data__{name}__oof.npy",
    pred_dir / f"H_data__{name}__test.npy",
    pred_dir / f"H_data__{name}__meta.json",
]

if all(path.exists() for path in expected_files):
    print(f"[건너뜀] {name}: 예측 파일 3개 존재")

else:
    # 전체 학습 데이터로 최종 모델 학습
    final_E_LR = clone(pipeline_E_LR)
    final_E_LR.fit(X, y_encoded)

    # Test 예측 확률 생성
    test_proba_E = final_E_LR.predict_proba(test_features)

    # OOF / Test 확률 크기 검증
    assert cv_E_LR["oof_proba"].shape == (
        len(X), len(label_encoder.classes_)
    )
    assert test_proba_E.shape == (
        len(test_features), len(label_encoder.classes_)
    )

    # 클래스 순서 검증
    assert np.array_equal(
        final_E_LR.classes_,
        np.arange(len(label_encoder.classes_))
    )

    save_predictions(
        owner="H_data",
        name=name,
        oof_proba=cv_E_LR["oof_proba"],
        test_proba=test_proba_E,
        classes=label_encoder.classes_,
        cv=cv_E_LR,
        notes="Grouped 5-Fold, LR C=0.1 balanced, AA pairs + gene truncation"
    )

    print(f"[저장 완료] {name}")

예측 저장: H_data__E_aa_pairs_truncation_group_lr (CV 0.4919) → experiments/preds/
[저장 완료] E_aa_pairs_truncation_group_lr


#### 번회 포함 비교표

In [ ]:
cv_summary = pd.DataFrame([
    ["A (freq 5)", "Multi-hot (freq ≥5)", "XGBoost", "기존 Fold", 0.1590],
    ["A (freq 3)", "Multi-hot (freq ≥3)", "XGBoost", "기존 Fold", 0.1630],
    ["B", "Binary", "XGBoost", "기존 Fold", 0.3200],
    ["C", "B + Mutation Count", "XGBoost", "기존 Fold", 0.3663],
    ["D", "C + Mutation Types", "XGBoost", "기존 Fold", 0.3836],
    ["D + Multi-hot (freq 5)", "D + Multi-hot (freq ≥5)", "XGBoost", "기존 Fold", 0.3867],
    ["D + Multi-hot (freq 3)", "D + Multi-hot (freq ≥3)", "XGBoost", "기존 Fold", 0.3851],
    ["D + AA (freq 16)", "D + Amino Acid Pairs (freq ≥16)", "XGBoost", "기존 Fold", 0.4271],
    ["E", "D 기반 + AA Pairs + Truncation", "LogisticRegression", "새 Group Fold", cv_E_LR["mean"]],
], columns=["실험", "피처 구성", "모델", "CV 기준", "Macro F1"])

display(cv_summary)

,실험,피처 구성,모델,CV 기준,Macro F1
0,A (freq 5),Multi-hot (freq ≥5),XGBoost,기존 Fold,0.159000
1,A (freq 3),Multi-hot (freq ≥3),XGBoost,기존 Fold,0.163000
2,B,Binary,XGBoost,기존 Fold,0.320000
3,C,B + Mutation Count,XGBoost,기존 Fold,0.366300
4,D,C + Mutation Types,XGBoost,기존 Fold,0.383600
5,D + Multi-hot (freq 5),D + Multi-hot (freq ≥5),XGBoost,기존 Fold,0.386700
6,D + Multi-hot (freq 3),D + Multi-hot (freq ≥3),XGBoost,기존 Fold,0.385100
7,D + AA (freq 16),D + Amino Acid Pairs (freq ≥16),XGBoost,기존 Fold,0.427100
8,E,D 기반 + AA Pairs + Truncation,LogisticRegression,새 Group Fold,0.491935


## 5. 전처리

### 5-1. 전처리 전 확인사항

In [ ]:
# 변이 문자열의 실제 구조 확인

gene_cols = train.columns.drop(["ID", "SUBCLASS"])

# WT를 제외한 실제 변이 값
mutations = train[gene_cols].stack()
mutations = mutations[mutations != "WT"]

print("전체 변이 기록 수:", len(mutations))
print("고유 변이 문자열 수:", mutations.nunique())

print("\n변이 문자열 빈도 TOP 30")
display(mutations.value_counts().head(30))

print("\n변이 문자열 예시 30개")
print(mutations.drop_duplicates().head(30).tolist())

전체 변이 기록 수: 218893
고유 변이 문자열 수: 118846

변이 문자열 빈도 TOP 30


R132H     349
V600E     305
E545K     135
H1047R    128
R273C      82
E542K      68
R175H      66
R273H      60
R132C      59
R248Q      49
R213*      45
R248W      45
L42L       39
R282W      34
R196*      32
Y220C      32
K16fs      31
R80*       31
L35L       28
L100L      27
A2A        25
Q61R       25
Q269Q      25
H179R      25
L45L       24
P423P      24
R306*      24
G245S      24
E110K      24
L159L      23
Name: count, dtype: int64


변이 문자열 예시 30개
['D623D', 'F736C', 'P117R', 'R305S', 'N23K', 'L192V', 'G592G', 'K277K', 'A699T', 'L111L', 'I136M', 'A219T', 'E237*', 'A746D', 'D465D', 'N1445D', 'T218T', 'G764G', 'E660Q', 'E291A', 'I382M', 'V412A', 'P222R', 'R104H', 'P17R', 'T1115M', 'L1854fs', 'R36H', 'R69Q', 'A372T']


In [ ]:
# 변이 유형별 분포 및 미분류 값 확인

# 기존 mutations Series 활용
m = mutations.astype(str)

# 기본값: other
mutation_type = pd.Series("other", index=m.index)

# Frameshift
mutation_type.loc[
    m.str.contains("fs", case=False, regex=False)
] = "frameshift"

# Nonsense
mutation_type.loc[
    m.str.fullmatch(r"[A-Z]\d+\*")
] = "nonsense"

# 일반 아미노산 치환 형태
standard = m.str.extract(r"^([A-Z])(\d+)([A-Z])$")

valid = standard.notna().all(axis=1)

# Synonymous
synonymous = valid & (standard[0] == standard[2])
mutation_type.loc[synonymous] = "synonymous"

# Missense
missense = valid & (standard[0] != standard[2])
mutation_type.loc[missense] = "missense"

print("변이 유형별 기록 수")
display(mutation_type.value_counts())

print("\n변이 유형별 비율 (%)")
display((mutation_type.value_counts(normalize=True) * 100).round(2))

print("\n미분류 변이 문자열 TOP 30")
display(m[mutation_type == "other"].value_counts().head(30))

변이 유형별 기록 수


missense      127403
synonymous     51012
other          21155
nonsense        9781
frameshift      9542
Name: count, dtype: int64


변이 유형별 비율 (%)


missense      58.20
synonymous    23.30
other          9.66
nonsense       4.47
frameshift     4.36
Name: proportion, dtype: float64


미분류 변이 문자열 TOP 30


L26V L24V            19
V600E V600E          11
S8S S6A               9
S43G P45P             7
C19W R5R              6
G73G G81G             5
E726K H1047R          4
S372S K373E           4
H1047R H1047R         4
P534P E556K S557T     3
E545K E726K           3
E216K E216K           3
N345K N345K           3
E453K E545K           3
R876* R1450*          3
S183F S183F           3
L279L L279L           3
P11S P11S             3
P80L P80L             3
E28K E28K             3
E545K E545K           3
E556K S557T           3
G823A I824F           2
P45S K159*            2
V241L R245L           2
R130* R233*           2
468_469LG>F*          2
Q347* I346N           2
R282W R213*           2
312_313QY>HH          2
Name: count, dtype: int64

- other 9.66%를 그냥 기타 변이로 처리하면 유용한 정보를 잃을 수 있음.
- other 셀: 하나의 유전자 셀에 여러 변이 표기가 공백으로 연결되어 있는 걍우가 있으며, 468_469LG>F*, 312_313QY>HH처럼 단순한 아미노산 치환 패턴으로 분류할 수 없는 복합 표기도 존재함.

In [49]:
# 공백으로 분리한 변이 유형 재분석

# 하나의 셀에 기록된 변이를 공백 기준으로 분리
mutation_tokens = mutations.astype(str).str.split().explode()

# 유형 분류
token_type = pd.Series("other", index=mutation_tokens.index)

frameshift = mutation_tokens.str.contains(
    "fs", case=False, regex=False
)

nonsense = mutation_tokens.str.fullmatch(
    r"[A-Z]\d+\*"
)

standard = mutation_tokens.str.extract(
    r"^([A-Z])(\d+)([A-Z])$"
)

valid = standard.notna().all(axis=1)

synonymous = valid & (standard[0] == standard[2])
missense = valid & (standard[0] != standard[2])

# 중복 인덱스가 있을 수 있으므로 위치 기준으로 분류
import numpy as np

types = np.select(
    [
        frameshift.to_numpy(),
        nonsense.to_numpy(),
        synonymous.to_numpy(),
        missense.to_numpy()
    ],
    [
        "frameshift",
        "nonsense",
        "synonymous",
        "missense"
    ],
    default="other"
)

print("전체 변이 토큰 수:", len(mutation_tokens))

print("\n변이 유형별 분포")
display(pd.Series(types).value_counts())

print("\n미분류 비율 (%)")
print(round((types == "other").mean() * 100, 2))

print("\n미분류 변이 TOP 30")
display(
    mutation_tokens.reset_index(drop=True)[types == "other"]
    .value_counts()
    .head(30)
)

전체 변이 토큰 수: 255164

변이 유형별 분포


missense      164740
synonymous     66883
nonsense       13289
frameshift      9914
other            338
Name: count, dtype: int64


미분류 비율 (%)
0.13

미분류 변이 TOP 30


*261*             3
*1232*            3
*459*             3
468_469LG>F*      2
312_313QY>HH      2
82_83DA>EI        2
139_140MG>I*      2
292_293DP>ES      2
127_128KL>KL      2
929_930LL>LL      2
435_436EA>DS      2
*399*             2
*271*             2
711_712FL>FL      2
*220*             2
594_595ME>IK      2
*902*             2
*626*             2
*235*             2
1105_1106PL>PL    2
187_188PL>PL      2
1441_1442AL>AL    2
*837*             2
2069_2070WA>*T    2
*455*             2
*194*             2
*141*             2
*924*             2
*335*             2
264_265MG>I*      2
Name: count, dtype: int64

### 5-2. 파생변수 생성 (A,B,C,D,E,F)

In [50]:
# 샘플별 변이 유형 개수 생성

# mutation_tokens, types는 이전 셀에서 생성한 변수 사용

# 토큰별 변이 유형 데이터프레임
token_df = pd.DataFrame({
    "sample_idx": mutation_tokens.index.get_level_values(0),
    "mutation_type": types
})

# 샘플별 유형 개수 집계
type_counts = pd.crosstab(
    token_df["sample_idx"],
    token_df["mutation_type"]
)

# 모든 학습 샘플에 맞춰 정렬
type_cols = [
    "missense",
    "synonymous",
    "nonsense",
    "frameshift",
    "other"
]

type_counts = type_counts.reindex(
    index=train.index,
    columns=type_cols,
    fill_value=0
)

type_counts = type_counts.add_prefix("mutation_")

# 전체 변이 토큰 수
type_counts["mutation_total_tokens"] = (
    type_counts.sum(axis=1)
)

print("파생변수 크기:", type_counts.shape)
display(type_counts.head())

print("\n전체 토큰 수 검증:")
print("원본:", len(mutation_tokens))
print("집계:", type_counts["mutation_total_tokens"].sum())

파생변수 크기: (6201, 6)


mutation_type,mutation_missense,mutation_synonymous,mutation_nonsense,mutation_frameshift,mutation_other,mutation_total_tokens
0,17,7,1,0,0,25
1,9,5,0,3,0,17
2,79,39,6,0,0,124
3,3,3,1,0,0,7
4,23,11,0,0,0,34



전체 토큰 수 검증:
원본: 255164
집계: 255164


#### A(유전자별 이진 피처) , B(A+변이 유형별 개수)

In [51]:
# 기본 이진 피처 + 파생변수 결합

# 유전자 컬럼 선택
gene_cols = train.columns.drop(["ID", "SUBCLASS"])

# 유전자별 변이 존재 여부
X_binary = train[gene_cols].ne("WT").astype("uint8")

# 변이 유형별 파생변수 결합
X_features = pd.concat(
    [X_binary, type_counts],
    axis=1
)

print("기본 이진 피처:", X_binary.shape)
print("파생변수 결합 후:", X_features.shape)

print("\n결측치 수:", X_features.isna().sum().sum())
print("중복 컬럼 수:", X_features.columns.duplicated().sum())

display(X_features.iloc[:5, -6:])

기본 이진 피처: (6201, 4384)
파생변수 결합 후: (6201, 4390)

결측치 수: 0
중복 컬럼 수: 0


,mutation_missense,mutation_synonymous,mutation_nonsense,mutation_frameshift,mutation_other,mutation_total_tokens
0,17,7,1,0,0,25
1,9,5,0,3,0,17
2,79,39,6,0,0,124
3,3,3,1,0,0,7
4,23,11,0,0,0,34


#### C(B+변이 개수·로그·유형별 비율)

In [ ]:
# 추가 파생변수 생성

# 기존 X_features는 유지
X_features_v2 = X_features.copy()

# 변이가 기록된 유전자 개수
X_features_v2["mutation_gene_count"] = X_binary.sum(axis=1)

# 전체 토큰 수 로그 변환
X_features_v2["mutation_log_total_tokens"] = np.log1p(
    X_features_v2["mutation_total_tokens"]
)

# 변이 유형별 비율
for col in type_cols:
    X_features_v2[f"mutation_{col}_ratio"] = (
        X_features_v2[f"mutation_{col}"]
        / X_features_v2["mutation_total_tokens"].replace(0, np.nan)
    ).fillna(0)

print("기존 피처:", X_features.shape)
print("추가 후:", X_features_v2.shape)
print("결측치:", X_features_v2.isna().sum().sum())

display(X_features_v2.iloc[:5, -8:])

기존 피처: (6201, 4390)
추가 후: (6201, 4397)
결측치: 0


,mutation_total_tokens,mutation_gene_count,mutation_log_total_tokens,mutation_missense_ratio,mutation_synonymous_ratio,mutation_nonsense_ratio,mutation_frameshift_ratio,mutation_other_ratio
0,25,25,3.258097,0.680000,0.280000,0.040000,0.000000,0.0
1,17,17,2.890372,0.529412,0.294118,0.000000,0.176471,0.0
2,124,119,4.828314,0.637097,0.314516,0.048387,0.000000,0.0
3,7,7,2.079442,0.428571,0.428571,0.142857,0.000000,0.0
4,34,34,3.555348,0.676471,0.323529,0.000000,0.000000,0.0


#### D(B+변이 유전자 개수)

In [59]:
# 실험 B + 변이 유전자 개수
X_D = X_features.copy()

X_D["mutation_gene_count"] = X_binary.sum(axis=1)

print("실험 D 피처 크기:", X_D.shape)
print("결측치:", X_D.isna().sum().sum())
print("중복 컬럼:", X_D.columns.duplicated().sum())

실험 D 피처 크기: (6201, 4391)
결측치: 0
중복 컬럼: 0


#### E(B+전체 변이 토큰 수 로그 변환)

In [61]:
# 실험 B + 전체 변이 토큰 수 로그 변환
X_E = X_features.copy()

X_E["mutation_log_total_tokens"] = np.log1p(
    X_E["mutation_total_tokens"]
)

print("실험 E 피처 크기:", X_E.shape)
print("결측치:", X_E.isna().sum().sum())
print("중복 컬럼:", X_E.columns.duplicated().sum())

실험 E 피처 크기: (6201, 4391)
결측치: 0
중복 컬럼: 0


#### F(D+변이 유형별 비율 5개)

In [63]:
# 실험 D + 변이 유형별 비율 5개
X_F = X_D.copy()

ratio_cols = [
    f"mutation_{col}_ratio"
    for col in type_cols
]

X_F[ratio_cols] = X_features_v2[ratio_cols]

print("실험 F 피처 크기:", X_F.shape)
print("결측치:", X_F.isna().sum().sum())
print("중복 컬럼:", X_F.columns.duplicated().sum())

실험 F 피처 크기: (6201, 4396)
결측치: 0
중복 컬럼: 0


#### 타깃 변수와 피처 검증

In [53]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(train["SUBCLASS"])

print("샘플 수:", len(y))
print("클래스 수:", len(le.classes_))
print("클래스 목록:", le.classes_.tolist())

print("\n피처와 타깃 행 수 일치:")
print(
    len(X_binary) == len(X_features) == len(X_features_v2) == len(y)
)

print("\n피처 인덱스 일치:")
print(
    X_binary.index.equals(X_features.index)
    and X_features.index.equals(X_features_v2.index)
    and X_features_v2.index.equals(train.index)
)

샘플 수: 6201
클래스 수: 26
클래스 목록: ['ACC', 'BLCA', 'BRCA', 'CESC', 'COAD', 'DLBC', 'GBMLGG', 'HNSC', 'KIPAN', 'KIRC', 'LAML', 'LGG', 'LIHC', 'LUAD', 'LUSC', 'OV', 'PAAD', 'PCPG', 'PRAD', 'SARC', 'SKCM', 'STES', 'TGCT', 'THCA', 'THYM', 'UCEC']

피처와 타깃 행 수 일치:
True

피처 인덱스 일치:
True


### 5-3. 모델 학습 전 교차검증 설정 확인

In [54]:
import inspect
from src.validation import get_folds

print(inspect.signature(get_folds))
print(inspect.getsource(get_folds))

(y, X=None, classes=None, n_splits=5, seed=42)
def get_folds(
    y,
    X=None,
    classes=None,
    n_splits=N_SPLITS,
    seed=SEED
):
    """
    팀 공통 StratifiedGroupKFold.

    get_folds(y)
        -> train.csv를 읽어 원본 X와 클래스 순서를 가져옴

    get_folds(y, X=X, classes=le.classes_)
        -> 전달받은 X와 클래스를 사용

    두 방식은 동일한 Fold를 생성해야 함.
    """
    from .config import DATA_DIR, ID_COL, TARGET
    from .data_loader import split_xy, encode_target

    y = np.asarray(y).ravel()

    if X is None:
        # test.csv는 읽지 않고 train만 사용
        train = pd.read_csv(DATA_DIR / "train.csv")

        if len(y) != len(train):
            raise ValueError(
                f"y 길이({len(y)})와 "
                f"train 길이({len(train)})가 다릅니다."
            )

        X, y_raw = split_xy(train)
        _, le = encode_target(y_raw)

        if classes is None:
            classes = le.classes_

    else:
        if len(y) != len(X):
            raise ValueError(
                f"y 길이({len(y)})와 "
        

### 5-4. 실제 Fold 생성 및 확인

In [55]:
folds = get_folds(
    y,
    X=train[gene_cols],
    classes=le.classes_
)

print("Fold 수:", len(folds))

for i, (train_idx, val_idx) in enumerate(folds, 1):
    print(
        f"Fold {i}: "
        f"학습 {len(train_idx)}개 / "
        f"검증 {len(val_idx)}개"
    )

Fold 수: 5
Fold 1: 학습 4961개 / 검증 1240개
Fold 2: 학습 4960개 / 검증 1241개
Fold 3: 학습 4961개 / 검증 1240개
Fold 4: 학습 4961개 / 검증 1240개
Fold 5: 학습 4961개 / 검증 1240개


### 5-5. 파생변수 실험 A,B,C,D,E,F 성능 비교

#### A (유전자별 이진 피처) 학습 

In [56]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
import numpy as np

X_A = X_binary

scores_A = []

for fold, (train_idx, val_idx) in enumerate(folds, 1):

    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(le.classes_),
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_A.iloc[train_idx],
        y[train_idx]
    )

    preds = model.predict(X_A.iloc[val_idx])

    score = f1_score(
        y[val_idx],
        preds,
        average="macro",
        labels=np.arange(len(le.classes_)),
        zero_division=0
    )

    scores_A.append(score)

    print(f"Fold {fold}: Macro F1 = {score:.4f}")

print(f"\n실험 A 평균 Macro F1: {np.mean(scores_A):.4f}")
print(f"실험 A 표준편차: {np.std(scores_A):.4f}")

Fold 1: Macro F1 = 0.3328
Fold 2: Macro F1 = 0.3614
Fold 3: Macro F1 = 0.3177
Fold 4: Macro F1 = 0.3306
Fold 5: Macro F1 = 0.3463

실험 A 평균 Macro F1: 0.3378
실험 A 표준편차: 0.0149


#### B (A+변이 유형별 개수) 학습

In [57]:
X_B = X_features

scores_B = []

for fold, (train_idx, val_idx) in enumerate(folds, 1):

    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(le.classes_),
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_B.iloc[train_idx],
        y[train_idx]
    )

    preds = model.predict(X_B.iloc[val_idx])

    score = f1_score(
        y[val_idx],
        preds,
        average="macro",
        labels=np.arange(len(le.classes_)),
        zero_division=0
    )

    scores_B.append(score)

    print(f"Fold {fold}: Macro F1 = {score:.4f}")

print(f"\n실험 B 평균 Macro F1: {np.mean(scores_B):.4f}")
print(f"실험 B 표준편차: {np.std(scores_B):.4f}")
print(f"A 대비 변화: {np.mean(scores_B) - np.mean(scores_A):+.4f}")

Fold 1: Macro F1 = 0.3954
Fold 2: Macro F1 = 0.4193
Fold 3: Macro F1 = 0.3899
Fold 4: Macro F1 = 0.4258
Fold 5: Macro F1 = 0.4141

실험 B 평균 Macro F1: 0.4089
실험 B 표준편차: 0.0139
A 대비 변화: +0.0712


#### C (B+변이 개수·로그·유형별 비율) 학습

In [58]:
X_C = X_features_v2

scores_C = []

for fold, (train_idx, val_idx) in enumerate(folds, 1):

    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(le.classes_),
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_C.iloc[train_idx],
        y[train_idx]
    )

    preds = model.predict(X_C.iloc[val_idx])

    score = f1_score(
        y[val_idx],
        preds,
        average="macro",
        labels=np.arange(len(le.classes_)),
        zero_division=0
    )

    scores_C.append(score)

    print(f"Fold {fold}: Macro F1 = {score:.4f}")

print(f"\n실험 C 평균 Macro F1: {np.mean(scores_C):.4f}")
print(f"실험 C 표준편차: {np.std(scores_C):.4f}")
print(f"B 대비 변화: {np.mean(scores_C) - np.mean(scores_B):+.4f}")

Fold 1: Macro F1 = 0.3993
Fold 2: Macro F1 = 0.4150
Fold 3: Macro F1 = 0.3787
Fold 4: Macro F1 = 0.4021
Fold 5: Macro F1 = 0.4226

실험 C 평균 Macro F1: 0.4035
실험 C 표준편차: 0.0150
B 대비 변화: -0.0054


#### D(B+변이 유전자 개수) 학습

In [60]:
scores_D = []

for fold, (train_idx, val_idx) in enumerate(folds, 1):

    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(le.classes_),
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_D.iloc[train_idx],
        y[train_idx]
    )

    preds = model.predict(X_D.iloc[val_idx])

    score = f1_score(
        y[val_idx],
        preds,
        average="macro",
        labels=np.arange(len(le.classes_)),
        zero_division=0
    )

    scores_D.append(score)

    print(f"Fold {fold}: Macro F1 = {score:.4f}")

print(f"\n실험 D 평균 Macro F1: {np.mean(scores_D):.4f}")
print(f"실험 D 표준편차: {np.std(scores_D):.4f}")
print(f"B 대비 변화: {np.mean(scores_D) - np.mean(scores_B):+.4f}")

Fold 1: Macro F1 = 0.3985
Fold 2: Macro F1 = 0.4333
Fold 3: Macro F1 = 0.3876
Fold 4: Macro F1 = 0.4222
Fold 5: Macro F1 = 0.4087

실험 D 평균 Macro F1: 0.4101
실험 D 표준편차: 0.0163
B 대비 변화: +0.0011


#### E (B+전체 변이 토큰 수 로그 변환) 학습

In [62]:
scores_E = []

for fold, (train_idx, val_idx) in enumerate(folds, 1):

    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(le.classes_),
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_E.iloc[train_idx],
        y[train_idx]
    )

    preds = model.predict(X_E.iloc[val_idx])

    score = f1_score(
        y[val_idx],
        preds,
        average="macro",
        labels=np.arange(len(le.classes_)),
        zero_division=0
    )

    scores_E.append(score)

    print(f"Fold {fold}: Macro F1 = {score:.4f}")

print(f"\n실험 E 평균 Macro F1: {np.mean(scores_E):.4f}")
print(f"실험 E 표준편차: {np.std(scores_E):.4f}")
print(f"B 대비 변화: {np.mean(scores_E) - np.mean(scores_B):+.4f}")

Fold 1: Macro F1 = 0.3954
Fold 2: Macro F1 = 0.4193
Fold 3: Macro F1 = 0.3899
Fold 4: Macro F1 = 0.4258
Fold 5: Macro F1 = 0.4141

실험 E 평균 Macro F1: 0.4089
실험 E 표준편차: 0.0139
B 대비 변화: +0.0000


- E는 B와 5개 Fold 점수가 전부 동일

#### F (D+변이 유형별 비율 5개) 학습

In [64]:
scores_F = []

for fold, (train_idx, val_idx) in enumerate(folds, 1):

    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(le.classes_),
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_F.iloc[train_idx],
        y[train_idx]
    )

    preds = model.predict(X_F.iloc[val_idx])

    score = f1_score(
        y[val_idx],
        preds,
        average="macro",
        labels=np.arange(len(le.classes_)),
        zero_division=0
    )

    scores_F.append(score)

    print(f"Fold {fold}: Macro F1 = {score:.4f}")

print(f"\n실험 F 평균 Macro F1: {np.mean(scores_F):.4f}")
print(f"실험 F 표준편차: {np.std(scores_F):.4f}")
print(f"D 대비 변화: {np.mean(scores_F) - np.mean(scores_D):+.4f}")

Fold 1: Macro F1 = 0.3993
Fold 2: Macro F1 = 0.4150
Fold 3: Macro F1 = 0.3787
Fold 4: Macro F1 = 0.4021
Fold 5: Macro F1 = 0.4226

실험 F 평균 Macro F1: 0.4035
실험 F 표준편차: 0.0150
D 대비 변화: -0.0065


✅파생변수 실험에서 확인한 점
- 변이 유형별 개수: 가장 큰 성능 개선을 보임.
- 변이 유전자 개수: 소폭 개선됐지만 효과가 불확실함.
- 로그 변환: 추가 효과가 없음.
- 변이 유형별 비율: 이번 조합에서는 성능이 하락함.

### 5-6. 유전자와 아미노산 변이의 조합 피처 활용 - G, H 파생변수 생성 및 학습

In [66]:
# 현재 변이 조합 빈도 확인

# 유전자명 + 변이 토큰 조합 생성
mutation_pairs = (
    mutation_tokens.index.get_level_values(1).astype(str)
    + "_"
    + mutation_tokens.astype(str).to_numpy()
)

pair_counts = pd.Series(mutation_pairs).value_counts()

print("전체 고유 유전자-변이 조합:", len(pair_counts))
print("3회 이상 등장:", (pair_counts >= 3).sum())
print("5회 이상 등장:", (pair_counts >= 5).sum())
print("10회 이상 등장:", (pair_counts >= 10).sum())

display(pair_counts.head(20))

전체 고유 유전자-변이 조합: 226795
3회 이상 등장: 2797
5회 이상 등장: 452
10회 이상 등장: 122


IDH1_R132H       345
BRAF_V600E       332
PIK3CA_E545K     155
PIK3CA_H1047R    154
TP53_R273C        93
PIK3CA_E542K      76
TP53_R175H        70
TP53_R273H        63
TP53_R248Q        54
TP53_R248W        50
TP53_R213*        47
IDH1_R132C        46
TP53_Y220C        41
TP53_R196*        35
TP53_R282W        35
RPL22_K16fs       29
TP53_H179R        28
PTEN_R130Q        26
LRIG1_L24V        25
PIK3CA_H1047L     25
Name: count, dtype: int64

In [67]:
# 유전자-변이 조합 인코딩 함수 생성

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd

class MutationPairEncoder(BaseEstimator, TransformerMixin):

    def __init__(self, min_frequency=5):
        self.min_frequency = min_frequency

    def _extract_pairs(self, X):
        pairs = []

        for _, row in X.iterrows():
            sample_pairs = []

            for gene, value in row.items():
                if value == "WT" or pd.isna(value):
                    continue

                for mutation in str(value).split():
                    sample_pairs.append(f"{gene}_{mutation}")

            pairs.append(set(sample_pairs))

        return pairs

    def fit(self, X, y=None):
        pairs = self._extract_pairs(X)

        counts = pd.Series([
            pair
            for sample in pairs
            for pair in sample
        ]).value_counts()

        self.selected_pairs_ = set(
            counts[counts >= self.min_frequency].index
        )

        self.mlb_ = MultiLabelBinarizer(
            classes=sorted(self.selected_pairs_),
            sparse_output=True
        )
        self.mlb_.fit([[]])

        return self

    def transform(self, X):
        pairs = self._extract_pairs(X)

        filtered_pairs = [
            sample & self.selected_pairs_
            for sample in pairs
        ]

        return self.mlb_.transform(filtered_pairs)

In [68]:
# 변이 조합 인코딩 테스트

# Fold 1 가져오기
train_idx, val_idx = folds[0]

# 변이 조합 인코더 생성
pair_encoder = MutationPairEncoder(min_frequency=5)

# 학습 Fold에서만 변이 조합 선택
X_pair_train = pair_encoder.fit_transform(
    train.iloc[train_idx][gene_cols]
)

# 검증 Fold에 동일한 변이 조합 적용
X_pair_val = pair_encoder.transform(
    train.iloc[val_idx][gene_cols]
)

print("선택된 변이 조합 수:", len(pair_encoder.selected_pairs_))
print("학습 피처 크기:", X_pair_train.shape)
print("검증 피처 크기:", X_pair_val.shape)
print("학습 데이터 형식:", type(X_pair_train).__name__)

선택된 변이 조합 수: 303
학습 피처 크기: (4961, 303)
검증 피처 크기: (1240, 303)
학습 데이터 형식: csr_matrix


#### G (D + Multi-Hot), min_frequency=5

In [69]:
# G (D + Multi-Hot)
from scipy.sparse import hstack, csr_matrix

scores_G = []

for fold, (train_idx, val_idx) in enumerate(folds, 1):

    # 1. 원본 변이 데이터
    X_raw_train = train.iloc[train_idx][gene_cols]
    X_raw_val = train.iloc[val_idx][gene_cols]

    # 2. Fold별 변이 조합 인코딩
    encoder = MutationPairEncoder(min_frequency=5)

    X_pair_train = encoder.fit_transform(X_raw_train)
    X_pair_val = encoder.transform(X_raw_val)

    # 3. 기존 D 피처 + Multi-Hot 결합
    X_train_G = hstack([
        csr_matrix(X_D.iloc[train_idx].to_numpy()),
        X_pair_train
    ], format="csr")

    X_val_G = hstack([
        csr_matrix(X_D.iloc[val_idx].to_numpy()),
        X_pair_val
    ], format="csr")

    # 4. XGBoost 학습
    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(le.classes_),
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_G, y[train_idx])

    preds = model.predict(X_val_G)

    score = f1_score(
        y[val_idx],
        preds,
        average="macro",
        labels=np.arange(len(le.classes_)),
        zero_division=0
    )

    scores_G.append(score)

    print(
        f"Fold {fold}: Macro F1 = {score:.4f} "
        f"| Multi-Hot 피처 {X_pair_train.shape[1]}개"
    )

print(f"\n실험 G 평균 Macro F1: {np.mean(scores_G):.4f}")
print(f"실험 G 표준편차: {np.std(scores_G):.4f}")
print(f"D 대비 변화: {np.mean(scores_G) - np.mean(scores_D):+.4f}")


Fold 1: Macro F1 = 0.4112 | Multi-Hot 피처 303개
Fold 2: Macro F1 = 0.4455 | Multi-Hot 피처 290개
Fold 3: Macro F1 = 0.3752 | Multi-Hot 피처 301개
Fold 4: Macro F1 = 0.4364 | Multi-Hot 피처 311개
Fold 5: Macro F1 = 0.4288 | Multi-Hot 피처 300개

실험 G 평균 Macro F1: 0.4194
실험 G 표준편차: 0.0248
D 대비 변화: +0.0093


#### H (D + Multi-Hot), min_frequency=3

In [70]:
scores_H = []

for fold, (train_idx, val_idx) in enumerate(folds, 1):

    X_raw_train = train.iloc[train_idx][gene_cols]
    X_raw_val = train.iloc[val_idx][gene_cols]

    encoder = MutationPairEncoder(min_frequency=3)

    X_pair_train = encoder.fit_transform(X_raw_train)
    X_pair_val = encoder.transform(X_raw_val)

    X_train_H = hstack([
        csr_matrix(X_D.iloc[train_idx].to_numpy()),
        X_pair_train
    ], format="csr")

    X_val_H = hstack([
        csr_matrix(X_D.iloc[val_idx].to_numpy()),
        X_pair_val
    ], format="csr")

    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(le.classes_),
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_H, y[train_idx])

    preds = model.predict(X_val_H)

    score = f1_score(
        y[val_idx],
        preds,
        average="macro",
        labels=np.arange(len(le.classes_)),
        zero_division=0
    )

    scores_H.append(score)

    print(
        f"Fold {fold}: Macro F1 = {score:.4f} "
        f"| Multi-Hot 피처 {X_pair_train.shape[1]}개"
    )

print(f"\n실험 H 평균 Macro F1: {np.mean(scores_H):.4f}")
print(f"실험 H 표준편차: {np.std(scores_H):.4f}")
print(f"G 대비 변화: {np.mean(scores_H) - np.mean(scores_G):+.4f}")

Fold 1: Macro F1 = 0.4081 | Multi-Hot 피처 1379개
Fold 2: Macro F1 = 0.4552 | Multi-Hot 피처 1493개
Fold 3: Macro F1 = 0.3793 | Multi-Hot 피처 1471개
Fold 4: Macro F1 = 0.4365 | Multi-Hot 피처 1582개
Fold 5: Macro F1 = 0.4219 | Multi-Hot 피처 1528개

실험 H 평균 Macro F1: 0.4202
실험 H 표준편차: 0.0257
G 대비 변화: +0.0008


### 5-7. 아미노산 치환 패턴 피처 활용 - I, J 파생변수 생성 및 학습

In [71]:
#  아미노산 치환 패턴 빈도 분석

# 미스센스 변이 토큰만 선택
missense_tokens = mutation_tokens[
    pd.Series(types, index=mutation_tokens.index).eq("missense")
]

# 시작 아미노산과 변경된 아미노산 추출
aa_pairs = missense_tokens.str.extract(
    r"^([A-Z])\d+([A-Z])$"
)

# 치환 패턴 생성
aa_pair_labels = (
    aa_pairs[0] + ">" + aa_pairs[1]
)

aa_pair_counts = aa_pair_labels.value_counts()

print("전체 미스센스 변이 토큰 수:", len(missense_tokens))
print("고유 아미노산 치환 패턴 수:", len(aa_pair_counts))

display(aa_pair_counts.head(20))

전체 미스센스 변이 토큰 수: 164740
고유 아미노산 치환 패턴 수: 241


E>K    9864
R>Q    5851
R>C    5644
R>H    5361
D>N    5135
P>S    5033
A>T    4837
A>V    4784
P>L    4421
S>F    4014
R>W    3499
S>L    3096
G>E    2784
K>N    2772
G>R    2654
E>D    2504
L>F    2419
E>Q    2393
F>L    2257
V>I    2085
Name: count, dtype: int64

⬆️ 해석
- 유전자와 변이 위치가 달라도 동일한 아미노산 치환 패턴을 공유함.
- 그러나 E>K가 자주 등장한다고 해서 특정 암종을 잘 구분한다는 뜻은 아니며, 모델 성능으로 확인해야 함.

In [72]:
# 샘플별 아미노산 치환 패턴 피처 생성

# 미스센스 토큰의 샘플 인덱스와 치환 패턴 결합
aa_pair_df = pd.DataFrame({
    "sample_idx": missense_tokens.index.get_level_values(0),
    "aa_pair": aa_pair_labels.to_numpy()
})

# 샘플별 치환 패턴 개수
aa_pair_features = pd.crosstab(
    aa_pair_df["sample_idx"],
    aa_pair_df["aa_pair"]
)

# 전체 학습 샘플 순서에 맞추기
aa_pair_features = aa_pair_features.reindex(
    index=train.index,
    fill_value=0
)

# 기존 피처와 컬럼명 충돌 방지
aa_pair_features = aa_pair_features.add_prefix("aa_pair_")

print("아미노산 치환 피처 크기:", aa_pair_features.shape)
print("결측치:", aa_pair_features.isna().sum().sum())
print("전체 치환 개수:", aa_pair_features.to_numpy().sum())

아미노산 치환 피처 크기: (6201, 241)
결측치: 0
전체 치환 개수: 164740


#### I (H+아미노산 치환 패턴)

In [73]:
scores_I = []

for fold, (train_idx, val_idx) in enumerate(folds, 1):

    # 1. 원본 변이 데이터
    X_raw_train = train.iloc[train_idx][gene_cols]
    X_raw_val = train.iloc[val_idx][gene_cols]

    # 2. Fold별 Multi-Hot 인코딩
    encoder = MutationPairEncoder(min_frequency=3)

    X_pair_train = encoder.fit_transform(X_raw_train)
    X_pair_val = encoder.transform(X_raw_val)

    # 3. 기존 D + Multi-Hot + 아미노산 치환 피처
    X_train_I = hstack([
        csr_matrix(X_D.iloc[train_idx].to_numpy()),
        X_pair_train,
        csr_matrix(aa_pair_features.iloc[train_idx].to_numpy())
    ], format="csr")

    X_val_I = hstack([
        csr_matrix(X_D.iloc[val_idx].to_numpy()),
        X_pair_val,
        csr_matrix(aa_pair_features.iloc[val_idx].to_numpy())
    ], format="csr")

    # 4. XGBoost 학습
    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(le.classes_),
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_I, y[train_idx])

    preds = model.predict(X_val_I)

    score = f1_score(
        y[val_idx],
        preds,
        average="macro",
        labels=np.arange(len(le.classes_)),
        zero_division=0
    )

    scores_I.append(score)

    print(
        f"Fold {fold}: Macro F1 = {score:.4f} "
        f"| 전체 피처 {X_train_I.shape[1]}개"
    )

print(f"\n실험 I 평균 Macro F1: {np.mean(scores_I):.4f}")
print(f"실험 I 표준편차: {np.std(scores_I):.4f}")
print(f"H 대비 변화: {np.mean(scores_I) - np.mean(scores_H):+.4f}")

Fold 1: Macro F1 = 0.4556 | 전체 피처 6011개
Fold 2: Macro F1 = 0.4794 | 전체 피처 6125개
Fold 3: Macro F1 = 0.4557 | 전체 피처 6103개
Fold 4: Macro F1 = 0.4864 | 전체 피처 6214개
Fold 5: Macro F1 = 0.4728 | 전체 피처 6160개

실험 I 평균 Macro F1: 0.4700
실험 I 표준편차: 0.0125
H 대비 변화: +0.0498


#### J (D + 아미노산 치환 패턴)

In [74]:
scores_J = []

for fold, (train_idx, val_idx) in enumerate(folds, 1):

    # 기존 D + 아미노산 치환 패턴
    X_train_J = hstack([
        csr_matrix(X_D.iloc[train_idx].to_numpy()),
        csr_matrix(aa_pair_features.iloc[train_idx].to_numpy())
    ], format="csr")

    X_val_J = hstack([
        csr_matrix(X_D.iloc[val_idx].to_numpy()),
        csr_matrix(aa_pair_features.iloc[val_idx].to_numpy())
    ], format="csr")

    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(le.classes_),
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_J, y[train_idx])

    preds = model.predict(X_val_J)

    score = f1_score(
        y[val_idx],
        preds,
        average="macro",
        labels=np.arange(len(le.classes_)),
        zero_division=0
    )

    scores_J.append(score)

    print(f"Fold {fold}: Macro F1 = {score:.4f}")

print(f"\n실험 J 평균 Macro F1: {np.mean(scores_J):.4f}")
print(f"실험 J 표준편차: {np.std(scores_J):.4f}")
print(f"I 대비 변화: {np.mean(scores_J) - np.mean(scores_I):+.4f}")

Fold 1: Macro F1 = 0.4453
Fold 2: Macro F1 = 0.4763
Fold 3: Macro F1 = 0.4354
Fold 4: Macro F1 = 0.4696
Fold 5: Macro F1 = 0.4701

실험 J 평균 Macro F1: 0.4593
실험 J 표준편차: 0.0160
I 대비 변화: -0.0106


⬆️ 해석
- 아미노산 치환 패턴의 기여가 큼. 
- Multi-Hot도 추가적인 효과가 있음. 
- 두 피처를 결합한 I가 가장 좋은 결과를 보임. 

### 5-8. 피처 중요도 분석

In [75]:
# 실험 I의 Fold 1 모델 재학습 및 중요도 추출

# Fold 1 데이터
train_idx, val_idx = folds[0]

# Multi-Hot 인코딩
encoder = MutationPairEncoder(min_frequency=3)

X_pair_train = encoder.fit_transform(
    train.iloc[train_idx][gene_cols]
)

# 전체 피처 결합
X_train_I = hstack([
    csr_matrix(X_D.iloc[train_idx].to_numpy()),
    X_pair_train,
    csr_matrix(aa_pair_features.iloc[train_idx].to_numpy())
], format="csr")

# 피처 이름 생성
feature_names_I = (
    list(X_D.columns)
    + ["pair_" + p for p in encoder.mlb_.classes_]
    + list(aa_pair_features.columns)
)

# 모델 학습
model_I = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softprob",
    num_class=len(le.classes_),
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

model_I.fit(X_train_I, y[train_idx])

# 피처 중요도
importance_I = pd.DataFrame({
    "feature": feature_names_I,
    "importance": model_I.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_I.head(20))

,feature,importance
3771,SOWAHC,0.024436
2244,LRIG1,0.009316
251,APC,0.007889
5260,pair_PIK3CA_H1047R,0.007499
5000,pair_LRIG1_L24V,0.007138
4539,pair_BRAF_V600E,0.006761
4280,VHL,0.005941
1888,IDH1,0.005501
3223,PTEN,0.004425
4903,pair_IDH1_R132H,0.004422


⬆️ 해석
- 상위 20개 피처 중 아미노산 치환 패턴(aa_pair_)은 하나도 없음.
- SOWAHC가 가장 높다는 점은 추가 확인이 필요하며, 특정 암종과 강하게 연관된 것인지, 데이터셋의 특이한 패턴을 학습한 것인지는 아직 알 수 없음.

In [77]:
# 피처 그룹별 중요도 확인

# 피처 그룹 구분
importance_I["group"] = np.select(
    [
        importance_I["feature"].str.startswith("pair_"),
        importance_I["feature"].str.startswith("aa_pair_")
    ],
    [
        "Mutation Multi-Hot",
        "Amino Acid Substitution"
    ],
    default="Base Features"
)

# 그룹별 중요도 요약
group_importance = (
    importance_I
    .groupby("group")
    .agg(
        feature_count=("feature", "count"),
        total_importance=("importance", "sum"),
        mean_importance=("importance", "mean"),
        max_importance=("importance", "max")
    )
    .sort_values("total_importance", ascending=False)
)

display(group_importance)

,feature_count,total_importance,mean_importance,max_importance
group,,,,
Base Features,4391,0.827551,0.000188,0.024436
Mutation Multi-Hot,1379,0.090244,0.000065,0.007499
Amino Acid Substitution,241,0.082205,0.000341,0.002403


⬆️ 해석
- 아미노산 치환 패턴의 평균 중요도가 세 그룹 중 가장 높음.

In [78]:
# 중요한 아미노산 치환 패턴 확인

# 아미노산 치환 패턴 피처만 추출
aa_importance = importance_I[
    importance_I["group"] == "Amino Acid Substitution"
].copy()

# 중요도 상위 20개
display(
    aa_importance[
        ["feature", "importance"]
    ].head(20)
)

# 모델에서 사용된 아미노산 치환 피처 수
print(
    "중요도 0 초과 피처 수:",
    (aa_importance["importance"] > 0).sum()
)

print(
    "전체 아미노산 치환 피처 수:",
    len(aa_importance)
)

,feature,importance
5921,aa_pair_P>S,0.002403
5873,aa_pair_K>T,0.001905
5816,aa_pair_E>Q,0.001745
5953,aa_pair_S>F,0.001729
5836,aa_pair_G>L,0.001546
5983,aa_pair_V>E,0.001439
5940,aa_pair_R>L,0.001437
6006,aa_pair_Y>F,0.001408
5937,aa_pair_R>H,0.001066
5830,aa_pair_G>C,0.001030


중요도 0 초과 피처 수: 151
전체 아미노산 치환 피처 수: 241


In [80]:
# 상위 20개 아미노산 치환 패턴 표

display(
    aa_importance[
        ["feature", "importance"]
    ].head(20)
)

,feature,importance
5921,aa_pair_P>S,0.002403
5873,aa_pair_K>T,0.001905
5816,aa_pair_E>Q,0.001745
5953,aa_pair_S>F,0.001729
5836,aa_pair_G>L,0.001546
5983,aa_pair_V>E,0.001439
5940,aa_pair_R>L,0.001437
6006,aa_pair_Y>F,0.001408
5937,aa_pair_R>H,0.001066
5830,aa_pair_G>C,0.001030


⬆️ 해석
- 앞서 전체 변이 빈도를 분석했을 때 가장 흔했던 패턴은 E>K(9,864회)였으나, 중요도 순위에서는 15위임.
- 반면 P>S는 빈도 5,033회로 상대적으로 적지만 중요도는 1위임.
- 즉, 자주 발생하는 변이 패턴과 암종 분류에 유용한 변이 패턴은 반드시 일치하지 않음.

### 5-9. 상위 치환 패턴의 암종별 분포 분석 - L 파생변수 생성 및 학습

In [ ]:
# 중요도가 높았던 P>S 패턴부터 확인

# 분석 대상 패턴
target_pair = "aa_pair_P>S"

# 암종별 샘플 수와 패턴 보유 샘플 수
pair_analysis = pd.DataFrame({
    "SUBCLASS": train["SUBCLASS"],
    "has_pair": aa_pair_features[target_pair] > 0
})

pair_summary = (
    pair_analysis
    .groupby("SUBCLASS")
    .agg(
        total_samples=("has_pair", "size"),
        pair_samples=("has_pair", "sum")
    )
)

# 암종별 패턴 보유 비율
pair_summary["pair_rate"] = (
    pair_summary["pair_samples"]
    / pair_summary["total_samples"]
)

# 비율이 높은 순서로 정렬
pair_summary = pair_summary.sort_values(
    "pair_rate",
    ascending=False
)

display(pair_summary.head(15))

,total_samples,pair_samples,pair_rate
SUBCLASS,,,
SKCM,276,231,0.836957
LUSC,178,80,0.449438
BLCA,104,43,0.413462
STES,379,141,0.372032
LUAD,184,68,0.369565
DLBC,38,14,0.368421
UCEC,198,71,0.358586
HNSC,223,71,0.318386
COAD,223,59,0.264574


In [82]:
#  P>S 패턴의 암종별 평균 등장 횟수

# 암종별 P>S 등장 횟수 분석
pair_count_analysis = pd.DataFrame({
    "SUBCLASS": train["SUBCLASS"],
    "pair_count": aa_pair_features["aa_pair_P>S"]
})

pair_count_summary = (
    pair_count_analysis
    .groupby("SUBCLASS")
    .agg(
        total_samples=("pair_count", "size"),
        mean_count=("pair_count", "mean"),
        median_count=("pair_count", "median"),
        max_count=("pair_count", "max")
    )
    .sort_values("mean_count", ascending=False)
)

display(pair_count_summary.head(15))

,total_samples,mean_count,median_count,max_count
SUBCLASS,,,,
SKCM,276,11.213768,5.0,651
UCEC,198,1.510101,0.0,36
LUSC,178,0.887640,0.0,51
PAAD,120,0.866667,0.0,82
DLBC,38,0.789474,0.0,10
STES,379,0.778364,0.0,11
BLCA,104,0.721154,0.0,6
COAD,223,0.654709,0.0,14
LUAD,184,0.592391,0.0,5


⬆️ 해석
- SKCM에서 P>S 패턴이 다른 암종보다 훨씬 많이 등장함.
- 즉, SKCM은 일부 샘플에서만 P>S가 나타나는 것이 아니라, 상당수 샘플에서 반복적으로 관찰되는 특징이 있음.
- 최댓값 651회는 확인이 필요함.

In [84]:
# SKCM의 P>S 분포 확인

# SKCM 샘플의 P>S 등장 횟수
skcm_ps = aa_pair_features.loc[
    train["SUBCLASS"] == "SKCM",
    "aa_pair_P>S"
]

print("SKCM P>S 분포")
print(skcm_ps.describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
))

print("\nP>S 50회 초과 샘플 수:", (skcm_ps > 50).sum())
print("P>S 100회 초과 샘플 수:", (skcm_ps > 100).sum())

SKCM P>S 분포
count    276.000000
mean      11.213768
std       40.355529
min        0.000000
25%        2.000000
50%        5.000000
75%       11.000000
90%       23.000000
95%       30.500000
99%       66.000000
max      651.000000
Name: aa_pair_P>S, dtype: float64

P>S 50회 초과 샘플 수: 6
P>S 100회 초과 샘플 수: 1


⬆️ 해석
- 276개 SKCM 샘플 중 1개만 100회를 초과함.
- 최댓값 651회는 99% 분위수인 66회의 약 9.9배임.
- 따라서 평균 11.21회는 극단값의 영향을 받고 있으며, SKCM의 일반적인 샘플을 설명할 때는 중앙값 5회가 더 적절함.

In [87]:
# P>S 비율의 암종별 분포 확인

# 샘플별 전체 미스센스 변이 수
total_missense = aa_pair_features.sum(axis=1)

# P>S가 전체 미스센스 변이에서 차지하는 비율
ps_ratio = (
    aa_pair_features["aa_pair_P>S"]
    / total_missense.replace(0, np.nan)
).fillna(0)

# 암종별 비율 요약
ps_ratio_analysis = pd.DataFrame({
    "SUBCLASS": train["SUBCLASS"],
    "ps_ratio": ps_ratio
})

ps_ratio_summary = (
    ps_ratio_analysis
    .groupby("SUBCLASS")
    .agg(
        total_samples=("ps_ratio", "size"),
        mean_ratio=("ps_ratio", "mean"),
        median_ratio=("ps_ratio", "median")
    )
    .sort_values("mean_ratio", ascending=False)
)

display(ps_ratio_summary.head(15))

,total_samples,mean_ratio,median_ratio
SUBCLASS,,,
SKCM,276,0.089964,0.09397
SARC,198,0.024512,0.00000
PCPG,147,0.021542,0.00000
THCA,324,0.017283,0.00000
HNSC,223,0.016305,0.00000
LAML,158,0.015770,0.00000
UCEC,198,0.015149,0.00000
TGCT,124,0.015056,0.00000
PAAD,120,0.014929,0.00000


⬆️ 해석
- SKCM에서 전체 미스센스 변이 중 P>S 비율의 중앙값이 9.40%이나, 나머지 표시된 14개 암종에선 중앙값이 모두 0%임.
- 즉, SKCM에서 P>S가 많이 나타나는 현상은 단순히 전체 변이 수가 많기 때문만은 아닐 가능성을 뒷받침함.

In [88]:
# 아미노산 치환 패턴별 비율 피처 생성

# 샘플별 전체 미스센스 변이 수
total_missense = aa_pair_features.sum(axis=1)

# 각 아미노산 치환 패턴의 상대 비율
aa_pair_features_ratio = (
    aa_pair_features.div(
        total_missense.replace(0, np.nan),
        axis=0
    )
    .fillna(0)
)

# 컬럼명 구분
aa_pair_features_ratio = (
    aa_pair_features_ratio
    .add_prefix("ratio_")
)

print("비율 피처 크기:", aa_pair_features_ratio.shape)

print(
    "결측치:",
    aa_pair_features_ratio.isna().sum().sum()
)

print(
    "무한대 값:",
    np.isinf(aa_pair_features_ratio.to_numpy()).sum()
)

# P>S 비율 확인
display(
    aa_pair_features_ratio[
        ["ratio_aa_pair_P>S"]
    ].head(10)
)

비율 피처 크기: (6201, 241)
결측치: 0
무한대 값: 0


aa_pair,ratio_aa_pair_P>S
0,0.000000
1,0.000000
2,0.139241
3,0.000000
4,0.086957
5,0.006098
6,0.000000
7,0.000000
8,0.000000
9,0.166667


#### L (I + 아미노산 치환 비율)

In [89]:
scores_L = []

for fold, (train_idx, val_idx) in enumerate(folds, 1):

    # 1. 원본 변이 데이터
    X_raw_train = train.iloc[train_idx][gene_cols]
    X_raw_val = train.iloc[val_idx][gene_cols]

    # 2. Fold별 Multi-Hot 인코딩
    encoder = MutationPairEncoder(min_frequency=3)

    X_pair_train = encoder.fit_transform(X_raw_train)
    X_pair_val = encoder.transform(X_raw_val)

    # 3. D + Multi-Hot + 치환 횟수 + 치환 비율
    X_train_L = hstack([
        csr_matrix(X_D.iloc[train_idx].to_numpy()),
        X_pair_train,
        csr_matrix(aa_pair_features.iloc[train_idx].to_numpy()),
        csr_matrix(aa_pair_features_ratio.iloc[train_idx].to_numpy())
    ], format="csr")

    X_val_L = hstack([
        csr_matrix(X_D.iloc[val_idx].to_numpy()),
        X_pair_val,
        csr_matrix(aa_pair_features.iloc[val_idx].to_numpy()),
        csr_matrix(aa_pair_features_ratio.iloc[val_idx].to_numpy())
    ], format="csr")

    # 4. XGBoost 학습
    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(le.classes_),
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_L, y[train_idx])

    preds = model.predict(X_val_L)

    score = f1_score(
        y[val_idx],
        preds,
        average="macro",
        labels=np.arange(len(le.classes_)),
        zero_division=0
    )

    scores_L.append(score)

    print(
        f"Fold {fold}: Macro F1 = {score:.4f} "
        f"| 전체 피처 {X_train_L.shape[1]}개"
    )

print(f"\n실험 L 평균 Macro F1: {np.mean(scores_L):.4f}")
print(f"실험 L 표준편차: {np.std(scores_L):.4f}")
print(f"I 대비 변화: {np.mean(scores_L) - np.mean(scores_I):+.4f}")

Fold 1: Macro F1 = 0.4372 | 전체 피처 6252개
Fold 2: Macro F1 = 0.4434 | 전체 피처 6366개
Fold 3: Macro F1 = 0.4449 | 전체 피처 6344개
Fold 4: Macro F1 = 0.4698 | 전체 피처 6455개
Fold 5: Macro F1 = 0.4640 | 전체 피처 6401개

실험 L 평균 Macro F1: 0.4518
실험 L 표준편차: 0.0127
I 대비 변화: -0.0181


### 5-10. log1p 변환 피처 활용 - K 파생변수 생성 및 학습

In [85]:
# 아미노산 치환 횟수 로그 변환
aa_pair_features_log = np.log1p(aa_pair_features)

print("원본 피처 크기:", aa_pair_features.shape)
print("로그 변환 피처 크기:", aa_pair_features_log.shape)

print("로그 변환 후 결측치:",
      aa_pair_features_log.isna().sum().sum())

# P>S 변환 전후 비교
display(
    pd.DataFrame({
        "original": aa_pair_features["aa_pair_P>S"],
        "log1p": aa_pair_features_log["aa_pair_P>S"]
    }).sort_values("original", ascending=False).head(10)
)

원본 피처 크기: (6201, 241)
로그 변환 피처 크기: (6201, 241)
로그 변환 후 결측치: 0


,original,log1p
1369,651,6.480045
1789,82,4.418841
6153,76,4.343805
5660,75,4.330733
3624,63,4.158883
1883,55,4.025352
1232,55,4.025352
5748,51,3.951244
486,48,3.891820
3396,44,3.806662


#### K(I + 아미노산 치환 횟수 log1p 변환)

In [86]:
scores_K = []

for fold, (train_idx, val_idx) in enumerate(folds, 1):

    # 1. 원본 변이 데이터
    X_raw_train = train.iloc[train_idx][gene_cols]
    X_raw_val = train.iloc[val_idx][gene_cols]

    # 2. Fold별 Multi-Hot 인코딩
    encoder = MutationPairEncoder(min_frequency=3)

    X_pair_train = encoder.fit_transform(X_raw_train)
    X_pair_val = encoder.transform(X_raw_val)

    # 3. D + Multi-Hot + 로그 변환된 아미노산 치환 피처
    X_train_K = hstack([
        csr_matrix(X_D.iloc[train_idx].to_numpy()),
        X_pair_train,
        csr_matrix(aa_pair_features_log.iloc[train_idx].to_numpy())
    ], format="csr")

    X_val_K = hstack([
        csr_matrix(X_D.iloc[val_idx].to_numpy()),
        X_pair_val,
        csr_matrix(aa_pair_features_log.iloc[val_idx].to_numpy())
    ], format="csr")

    # 4. XGBoost 학습
    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(le.classes_),
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_K, y[train_idx])

    preds = model.predict(X_val_K)

    score = f1_score(
        y[val_idx],
        preds,
        average="macro",
        labels=np.arange(len(le.classes_)),
        zero_division=0
    )

    scores_K.append(score)

    print(f"Fold {fold}: Macro F1 = {score:.4f}")

print(f"\n실험 K 평균 Macro F1: {np.mean(scores_K):.4f}")
print(f"실험 K 표준편차: {np.std(scores_K):.4f}")
print(f"I 대비 변화: {np.mean(scores_K) - np.mean(scores_I):+.4f}")

Fold 1: Macro F1 = 0.4556
Fold 2: Macro F1 = 0.4794
Fold 3: Macro F1 = 0.4557
Fold 4: Macro F1 = 0.4863
Fold 5: Macro F1 = 0.4728

실험 K 평균 Macro F1: 0.4700
실험 K 표준편차: 0.0125
I 대비 변화: -0.0000


### 5-11. 파생변수 I의 클래스별 F1 분석

In [90]:
from sklearn.metrics import classification_report

oof_preds_I = np.full(len(y), -1, dtype=int)

for fold, (train_idx, val_idx) in enumerate(folds, 1):

    encoder = MutationPairEncoder(min_frequency=3)

    X_pair_train = encoder.fit_transform(
        train.iloc[train_idx][gene_cols]
    )
    X_pair_val = encoder.transform(
        train.iloc[val_idx][gene_cols]
    )

    X_train = hstack([
        csr_matrix(X_D.iloc[train_idx].to_numpy()),
        X_pair_train,
        csr_matrix(aa_pair_features.iloc[train_idx].to_numpy())
    ], format="csr")

    X_val = hstack([
        csr_matrix(X_D.iloc[val_idx].to_numpy()),
        X_pair_val,
        csr_matrix(aa_pair_features.iloc[val_idx].to_numpy())
    ], format="csr")

    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(le.classes_),
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y[train_idx])
    oof_preds_I[val_idx] = model.predict(X_val)

    print(f"Fold {fold} 완료")

assert np.all(oof_preds_I >= 0)

report_I = classification_report(
    y,
    oof_preds_I,
    labels=np.arange(len(le.classes_)),
    target_names=le.classes_,
    output_dict=True,
    zero_division=0
)

class_report_I = (
    pd.DataFrame(report_I)
    .T
    .loc[le.classes_, ["precision", "recall", "f1-score", "support"]]
    .sort_values("f1-score")
)

display(class_report_I)

Fold 1 완료
Fold 2 완료
Fold 3 완료
Fold 4 완료
Fold 5 완료


,precision,recall,f1-score,support
SARC,0.229885,0.101010,0.140351,198.0
KIRC,0.344660,0.212575,0.262963,334.0
PRAD,0.312903,0.364662,0.336806,266.0
PCPG,0.335570,0.340136,0.337838,147.0
THYM,0.283784,0.428571,0.341463,98.0
PAAD,0.432099,0.291667,0.348259,120.0
HNSC,0.400000,0.322870,0.357320,223.0
DLBC,1.000000,0.236842,0.382979,38.0
OV,0.359589,0.415020,0.385321,253.0
CESC,0.453782,0.348387,0.394161,155.0


⬆️ 해석
- SKCM은 매우 잘 분류하고 있음. (Precision: 0.9231, Recall: 0.8261, F1: 0.8719)
- SARC는 가장 개선이 필요한 암종임. (전체 198개 중 약 20개만 올바르게 예측, Recall: 0.1010, F1: 0.1404)
- KIRC는 334개 샘플이 있는데도 Recall이 0.2126에 불과함.

In [91]:
# 오분류가 발생하는 암종 확인
# SARC와 KIRC가 어떤 암종으로 잘못 예측되는지 확인

from sklearn.metrics import confusion_matrix

# 혼동행렬 생성
cm_I = confusion_matrix(
    y,
    oof_preds_I,
    labels=np.arange(len(le.classes_))
)

cm_df_I = pd.DataFrame(
    cm_I,
    index=le.classes_,
    columns=le.classes_
)

# SARC, KIRC의 오분류 결과
for cancer in ["SARC", "KIRC"]:

    errors = cm_df_I.loc[cancer].copy()

    # 정답으로 분류된 건수 제외
    errors[cancer] = 0

    print(f"\n[{cancer}] 주요 오분류 암종")

    display(
        errors.sort_values(ascending=False)
        .head(10)
        .to_frame(name="misclassified_count")
    )


[SARC] 주요 오분류 암종


,misclassified_count
BRCA,55
OV,26
KIPAN,21
PRAD,20
GBMLGG,17
PCPG,6
LAML,5
CESC,4
STES,4
TGCT,3



[KIRC] 주요 오분류 암종


,misclassified_count
KIPAN,171
BRCA,39
PRAD,23
GBMLGG,10
THCA,3
TGCT,3
OV,3
PCPG,2
THYM,2
LAML,2


⬆️ 해석
- SARC와 KIRC의 오분류 양상이 확실히 다름을 알 수 있음.

- SARC — 여러 암종으로 분산된 오분류
  - SARC는 BRCA로 가장 많이 오분류되지만, OV·KIPAN·PRAD·GBMLGG 등으로도 분산돼 있음. 이는 현재 모델이 SARC를 다른 암종과 구별할 만한 특징을 충분히 학습하지 못했을 가능성을 보여줌. 다만 SARC는 다양한 조직학적 아형을 포함하는 육종 계열이므로, 내부 이질성도 검토할 필요가 있음.

- KIRC — KIPAN으로 집중된 오분류
  - KIRC는 334개 중 171개(51.2%)가 KIPAN으로 잘못 분류됨. 상위 개념과 하위 개념의 관계로 생물학적 특성이 겹칠 가능성이 높음. 다만 이 데이터셋에서 KIPAN을 정확히 어떤 기준으로 정의했는지는 확인해야 함.

In [4]:
#  KIRC vs KIPAN 주요 변이 유전자 비교

# KIRC, KIPAN 샘플 선택
target_classes = ["KIRC", "KIPAN"]

# 암종별 유전자 변이 보유율 계산
gene_mutation_rates = (
    X_binary
    .groupby(train["SUBCLASS"])
    .mean()
    .loc[target_classes]
    .T
)

# KIRC - KIPAN 변이율 차이
gene_mutation_rates["rate_diff"] = (
    gene_mutation_rates["KIRC"]
    - gene_mutation_rates["KIPAN"]
)

# 절대 차이가 큰 유전자 상위 20개
top_diff_genes = (
    gene_mutation_rates
    .assign(abs_diff=lambda x: x["rate_diff"].abs())
    .sort_values("abs_diff", ascending=False)
    .head(20)
)

display(top_diff_genes)

SUBCLASS,KIRC,KIPAN,rate_diff,abs_diff
VHL,0.452096,0.299029,0.153067,0.153067
TP53,0.014970,0.046602,-0.031632,0.031632
CDC27,0.011976,0.034951,-0.022975,0.022975
MET,0.014970,0.036893,-0.021923,0.021923
DOCK4,0.002994,0.023301,-0.020307,0.020307
SKI,0.000000,0.019417,-0.019417,0.019417
CACNA1A,0.011976,0.029126,-0.017150,0.017150
NEFH,0.002994,0.019417,-0.016423,0.016423
PKD1,0.008982,0.025243,-0.016261,0.016261
MAML2,0.014970,0.031068,-0.016098,0.016098


⬆️ 해석
- KIRC와 KIPAN은 유전자 변이 보유율이 상당히 비슷하고, VHL만 상대적으로 큰 차이를 보임.

In [ ]:
# KIRC 정분류, 오분류 샘플의 VHL 변이율 비교

import pandas as pd

# 실제 KIRC 샘플만 선택
kirc_mask = train["SUBCLASS"].eq("KIRC")

# KIRC 샘플의 예측 결과
kirc_analysis = pd.DataFrame({
    "VHL_mutated": X_binary.loc[kirc_mask, "VHL"],
    "predicted_class": le.inverse_transform(
        oof_preds_I[kirc_mask.to_numpy()]
    )
})

# 예측 결과별 VHL 변이 보유율
vhl_by_prediction = (
    kirc_analysis
    .groupby("predicted_class")["VHL_mutated"]
    .agg(["count", "sum", "mean"])
    .sort_values("count", ascending=False)
)

display(vhl_by_prediction.head(10))

,count,sum,mean
predicted_class,,,
KIPAN,171,87,0.508772
KIRC,71,64,0.901408
BRCA,39,0,0.000000
PRAD,23,0,0.000000
GBMLGG,10,0,0.000000
OV,3,0,0.000000
TGCT,3,0,0.000000
THCA,3,0,0.000000
LAML,2,0,0.000000


⬆️ 해석
- KIRC를 예측할 때 모델이 VHL 변이 유무에 크게 영향을 받는 것으로 보이며, 특히 VHL 변이가 없는 KIRC 샘플에서 오분류가 집중되고 있음.
- 모델이 VHL 변이가 없는 KIRC 샘플을 다른 암종으로 분류하는 경향이 뚜렷함.
- 그러나 VHL 변이가 있어도 KIPAN으로 오분류된 KIRC가 87개나 존재함. 따라서 VHL만으로 KIRC와 KIPAN을 구분할 수 없음.

In [ ]:
# 💛🤍💛🤍💛🤍💛🤍자기전 돌릴코드 (체크포인트 저장)💛🤍💛🤍💛🤍
import joblib
import os

os.makedirs("cache", exist_ok=True)

joblib.dump({
    "train": train,
    "test": test,
    "gene_cols": gene_cols,
    "X_binary": X_binary,
    "X_D": X_D,
    "aa_pair_features": aa_pair_features,
    "aa_pair_features_ratio": aa_pair_features_ratio,
    "folds": folds,
    "y": y,
    "le": le,
    "oof_preds_I": oof_preds_I,
    "class_report_I": class_report_I,
    "scores_I": scores_I
}, "cache/feature_analysis_checkpoint.pkl")

print("저장 완료!")

저장 완료!


In [ ]:
# 💛🤍💛🤍💛🤍내일 실행 누르기 (체크포인트 복원)💛🤍💛🤍💛🤍
import joblib

checkpoint = joblib.load(
    "cache/feature_analysis_checkpoint.pkl"
)

globals().update(checkpoint)

print("변수 복원 완료!")

변수 복원 완료!


In [ ]:
# 💛🤍💛🤍💛🤍

print("train:", train.shape)
print("X_binary:", X_binary.shape)
print("X_D:", X_D.shape)
print("aa_pair_features:", aa_pair_features.shape)
print("folds:", len(folds))
print("OOF 예측:", len(oof_preds_I))

train: (6201, 4386)
X_binary: (6201, 4384)
X_D: (6201, 4391)
aa_pair_features: (6201, 241)
folds: 5
OOF 예측: 6201
